# 🏆 Job Hunt Agent — Resume Optimization
### Portfolio: Hallucination-Free Resume Optimization 

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                           SYSTEM ARCHITECTURE                                │
│                                                                              │
│  PDF Resume ──► [ResumeParser] ──► FactSheet (JSON) ──────────────────┐      │
│                                                                         │    │
│        Job Corpus ──► [FAISS Index] ──► top-K retrievals                │    │
│                               │                                         │    │
│                               ▼                                         ▼    │
│                  [ContrastiveRanker]              [ControlledOptimizer]      │
│                  Pairwise MLP (NEW: AUC/F1)      FactSheet-only bullets      │
│                    + MC Dropout CI                  + source_fact refs       │
│                               │                         │                    │
│                               └──────────┬──────────────┘                    │
│                                          ▼                                   │
│                          [LearnedHallucinationDetector] ◄── NEW ★            │
│                     Binary MLP trained on (bullet, source) pairs             │
│                     Outputs P(hallucination) + confidence gate               │
│                                          │                                   │
│                         ┌────────────────┴──────────────┐                    │
│                         ▼                                ▼                   │
│                  [PASS: confident]           [REJECT: regenerate]            │
│                         │                                                    │
│                         ▼                                                    │
│                 [GroundingEvaluator]                                         │
│       Halluc Rate · Faithfulness · AUC · F1 · Semantic Sim · JD Rel          │
└──────────────────────────────────────────────────────────────────────────────┘
```


## 📦 CELL 1 — Install Dependencies

In [1]:
import subprocess, sys
packages = [
    "google-genai>=1.0.0", "pymupdf", "pydantic>=2.0", "python-dotenv",
    "rich", "nest-asyncio", "sentence-transformers>=2.7.0",
    "scikit-learn>=1.4.0", "numpy>=1.26.0", "pandas>=2.0.0",
    "torch>=2.0.0", "faiss-cpu>=1.8.0", "matplotlib>=3.8.0", "scipy>=1.12.0",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("✅ All packages installed")


✅ All packages installed


## 🔑 CELL 2 — Config & Shared Utilities

In [2]:
import os, getpass, pathlib, logging, time, hashlib, json, re, math, random
from typing import Any
from dataclasses import dataclass, field
import numpy as np
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
try:
    from dotenv import load_dotenv; load_dotenv()
except ImportError:
    pass

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")

# ── Constants ────────────────────────────────────────────────────────────────
EMBED_MODEL_NAME     = "all-MiniLM-L6-v2"
LLM_MODEL_NAME       = "gemini-2.5-pro"
EMBED_DIM            = 384
TOP_K_RETRIEVAL      = 5
MC_DROPOUT_PASSES    = 20
RANKER_EPOCHS        = 12
CONFIDENCE_THRESHOLD = 0.55  
COSINE_FAITHFULNESS_THRESHOLD = 0.58
MAX_REGEN_ATTEMPTS   = 3       
HALLUC_EPOCHS        = 60       

CACHE_DIR            = "./cache"

pathlib.Path(CACHE_DIR).mkdir(exist_ok=True)
pathlib.Path("outputs").mkdir(exist_ok=True)

from rich.console import Console
from rich.panel import Panel
from rich.table import Table
console = Console()
logger  = logging.getLogger("job_hunt_agent")

# ── Latency profiler ─────────────────────────────────────────────────────────
LATENCY_LOG: list[dict] = []

def timed(label: str):
    def decorator(fn):
        def wrapper(*args, **kwargs):
            t0 = time.perf_counter()
            result = fn(*args, **kwargs)
            LATENCY_LOG.append({"component": label, "ms": round((time.perf_counter()-t0)*1000, 2)})
            return result
        return wrapper
    return decorator

print("✅ Config ready")


Paste your Gemini API key:  ········


✅ Config ready


---
## 🏗️ MODULE A — INGESTION LAYER
PDF → text, sentence-level embedding with disk cache. Hard error on missing PDF.

In [3]:
from sentence_transformers import SentenceTransformer

class EmbeddingCache:
    def __init__(self, cache_dir=CACHE_DIR):
        self._dir = pathlib.Path(cache_dir) / "embeddings"
        self._dir.mkdir(parents=True, exist_ok=True)
        self.hits = self.misses = 0

    def _key(self, text): return hashlib.sha256(text.encode()).hexdigest()[:16]

    def get(self, text):
        p = self._dir / f"{self._key(text)}.npy"
        if p.exists():
            self.hits += 1; return np.load(str(p))
        self.misses += 1; return None

    def set(self, text, vec):
        np.save(str(self._dir / f"{self._key(text)}.npy"), vec)

    @property
    def hit_rate(self):
        t = self.hits + self.misses; return self.hits / t if t else 0.0


class IngestionLayer:
    def __init__(self, model_name=EMBED_MODEL_NAME):
        console.print(f"Loading embedding model [cyan]{model_name}[/cyan]...")
        self._model = SentenceTransformer(model_name)
        self._cache = EmbeddingCache()
        console.print(f"✅ Embedding model ready (dim={self._model.get_sentence_embedding_dimension()})")

    # ─────────────────────────────────────────────────────────────
    # PDF → TEXT
    # ─────────────────────────────────────────────────────────────
    def parse_pdf_raw(self, path: str) -> str:
        """Raw PDF text — no cleaning. For LLM extraction only."""
        p = pathlib.Path(path)
        if not p.exists():
            raise FileNotFoundError(f"Resume PDF not found: {path}")
        import fitz
        doc = fitz.open(str(p))
        text = "\n".join(page.get_text() for page in doc)
        doc.close()
        return text.strip()

    def clean_for_embedding(self, text: str) -> str:
        """Strip glyphs, URLs, emails, symbols. For embeddings only."""
        import re
        text = re.sub(r'[§|ï¬•]+', ' ', text)
        text = re.sub(r'[^\x00-\x7F]+', ' ', text)
        text = re.sub(r'https?://\S+', '', text)
        text = re.sub(r'\S+@\S+\.\S+', '', text)
        text = re.sub(r'\b(Experience|Education|Skills|Projects|Publications|Awards|Summary)\b\s*\n?',
                      '', text)
        text = re.sub(r'\bCGPA\s*[\d./]+', '', text)
        text = re.sub(r'\s{2,}', ' ', text)
        return text.strip()

    def parse_pdf(self, path: str) -> str:
        p = pathlib.Path(path)
        if not p.exists():
            raise FileNotFoundError(f"Resume PDF not found: {path}")
        import fitz, re
        doc = fitz.open(str(p))
        text = "\n".join(page.get_text() for page in doc)
        doc.close()
        text = re.sub(r'[§|]+', ' ', text)
        text = re.sub(r'\b\w{1,2}\b\s*\|', '', text)
        text = re.sub(r'https?://\S+', '', text)
        text = re.sub(r'\S+@\S+\.\S+', '', text)
        text = re.sub(r'\s{2,}', ' ', text)
        return text.strip()

    # ─────────────────────────────────────────────────────────────
    # Sentence Processing Utilities
    # ─────────────────────────────────────────────────────────────
    def _split_sentences(self, text: str) -> list[str]:
        text = text.replace("\n", " ")
        sentences = re.split(r'(?<=[.!?])\s+', text)
        return [s.strip() for s in sentences if len(s.strip()) > 20]

    def _is_valid_sentence(self, s: str) -> bool:
        s_lower = s.lower()
        if any(x in s_lower for x in ["http", "www", ".com", ".io", "github", "linkedin"]):
            return False
        if re.search(r'\b(IAS|IPS|Secretary|Government|Ministry|Principal)\b', s):
            return False
        if len(s.split()) < 5:
            return False
        alpha_ratio = sum(c.isalnum() for c in s) / max(len(s), 1)
        if alpha_ratio < 0.6:
            return False
        return True

    def _process_sentences(self, text: str) -> list[str]:
        sents = self._split_sentences(text)
        sents = [s for s in sents if self._is_valid_sentence(s)]
        return sents if sents else [text[:512]]

    # ─────────────────────────────────────────────────────────────
    # EMBEDDING (Document Level)
    # ─────────────────────────────────────────────────────────────
    @timed("embed")
    def embed(self, text: str) -> np.ndarray:
        cached = self._cache.get(text)
        if cached is not None:
            return cached

        sents = self._process_sentences(text)

        vecs = self._model.encode(
            sents,
            normalize_embeddings=True,
            show_progress_bar=False,
            batch_size=64
        )

        weights = np.array([len(s.split()) for s in sents])
        weights = weights / weights.sum()

        pooled = (vecs * weights[:, None]).sum(axis=0)
        pooled = pooled / (np.linalg.norm(pooled) + 1e-8)

        self._cache.set(text, pooled)
        return pooled

    # ─────────────────────────────────────────────────────────────
    # EMBEDDING (Sentence Level for Explainability)
    # ─────────────────────────────────────────────────────────────
    # Section headers and metadata that should never appear as gradient drivers
    _EXPLAINER_BLOCK_TERMS = {
        "education", "cgpa", "gpa", "university", "institute", "college",
        "award", "awards", "conference", "publication", "publications",
        "m.tech", "b.tech", "b.e.", "m.e.", "ph.d", "phd",
        "ieee", "icmr", "nirt", "iit", "nit", "bits",
        "research dissemination", "gates foundation",
    }

    def embed_sentences(self, text: str) -> tuple[list[str], np.ndarray]:
        """
        Sentence extraction for explainability — strips education, publication,
        award, and contact lines BEFORE embedding so they can never appear as
        gradient drivers regardless of how they score.
        """
        import re

        text = text.replace("\n", " ")
        sentences = re.split(r'(?<=[.!?])\s+', text)

        clean_sents = []

        for s in sentences:
            s = s.strip()

            # Skip short / weak fragments
            if len(s) < 40:
                continue

            # Remove numeric-only / broken fragments
            if re.match(r'^[\d\s\./()%\-]+$', s):
                continue

            # Must contain enough real words
            if len(re.findall(r'\b[a-zA-Z]{3,}\b', s)) < 5:
                continue

            # Remove contact / link noise
            if any(x in s.lower() for x in ["github", "linkedin", "http", "www", "@"]):
                continue
            if re.search(r'\b(IAS|IPS|IFS|Secretary|Government|Ministry|'
                         r'Principal\s+Secretary|Commissioner|Director\s+General|'
                         r'Tamil\s+Nadu|Health\s+&|Family\s+Welfare)\b', s):
                continue
            # ── NEW: block education / publication / award sentences ──────────
            # Check against the hard block list — any matching term disqualifies
            # the sentence from appearing as a gradient driver.
            s_lower = s.lower()
            if any(term in s_lower for term in self._EXPLAINER_BLOCK_TERMS):
                continue

            # Additional pattern guards for degree lines and conference citations
            if re.search(r'\b(cgpa|gpa)\s*[:\-]?\s*[\d.]+', s_lower):
                continue
            if re.search(r'\b(20\d{2})\b', s) and any(
                t in s_lower for t in ["pune", "chennai", "delhi", "mumbai", "bangalore",
                                        "hyderabad", "diat", "nirt", "icmr"]
            ):
                continue  # location + year → institution/award line

            clean_sents.append(s)

        if not clean_sents:
            clean_sents = [text[:512]]

        vecs = self._model.encode(
            clean_sents,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        return clean_sents, vecs

    # ─────────────────────────────────────────────────────────────
    # Cache Stats
    # ─────────────────────────────────────────────────────────────
    def cache_stats(self):
        c = self._cache
        return f"Cache: {c.hits} hits / {c.misses} misses ({c.hit_rate:.1%} hit rate)"


INGESTION = IngestionLayer()
print("✅ Ingestion layer ready")


Loading embedding model all-MiniLM-L6-v2...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model ready (dim=384)

✅ Ingestion layer ready


---
## 🏗️ MODULE B — STRUCTURED FACT EXTRACTION (Stage 1)
**Why:** The LLM cannot hallucinate what it cannot see. Parsing the resume to a typed FactSheet first creates a hard boundary: only what's in the FactSheet can appear in the output.

In [4]:
import os, re, json
from typing import List, Optional
from pydantic import BaseModel, Field


# ─────────────────────────────────────────────────────────────
# Structured Models
# ─────────────────────────────────────────────────────────────

class Metric(BaseModel):
    value: str = ""
    context: str = ""
    source_sentence: str = ""


class Project(BaseModel):
    name: str = ""
    description: str = ""
    technologies: List[str] = Field(default_factory=list)
    metrics: List[Metric] = Field(default_factory=list)
    source_sentences: List[str] = Field(default_factory=list)


class Experience(BaseModel):
    title: str = ""
    organization: str = ""
    duration: str = ""
    bullets: List[str] = Field(default_factory=list)
    technologies: List[str] = Field(default_factory=list)
    metrics: List[Metric] = Field(default_factory=list)


class Contact(BaseModel):
    email: str = ""
    github: str = ""
    linkedin: str = ""
    location: str = ""


class FactSheet(BaseModel):
    candidate_name: str = ""
    contact: Contact = Field(default_factory=Contact)
    education: List[dict] = Field(default_factory=list)
    skills: List[str] = Field(default_factory=list)
    experiences: List[Experience] = Field(default_factory=list)
    projects: List[Project] = Field(default_factory=list)
    publications: List[str] = Field(default_factory=list)
    awards: List[str] = Field(default_factory=list)
    raw_text: str = ""


# ─────────────────────────────────────────────────────────────
# Prompt
# ─────────────────────────────────────────────────────────────

EXTRACTION_PROMPT = """You are a precise resume parser. Extract ALL facts from the resume below.

RULES (strictly enforced):
1. Extract ONLY information explicitly stated in the text.
2. Do NOT infer, extrapolate, or fill gaps.
3. For every metric (number, percentage, count), record the exact source_sentence.
4. List every skill mentioned — do not add any not present.
5. Return valid JSON matching the schema exactly.

RESUME TEXT:
{resume_text}

Return JSON:
{{
  "candidate_name": "...",
  "contact": {{"email":"...","github":"...","linkedin":"...","location":"..."}},
  "education": [{{"degree":"...","institution":"...","year":"...","gpa":"..."}}],
  "skills": ["skill1","skill2",...],
  "experiences": [{{
    "title":"...","organization":"...","duration":"...",
    "bullets":["exact bullet from resume",...],
    "technologies":["tech1",...],
    "metrics":[{{"value":"...","context":"...","source_sentence":"..."}}]
  }}],
  "projects": [{{
    "name":"...","description":"...","technologies":["..."],
    "metrics":[{{"value":"...","context":"...","source_sentence":"..."}}],
    "source_sentences":["..."]
  }}],
  "publications": ["..."],
  "awards": ["..."]
}}

Return ONLY the JSON. No markdown fences. No preamble.
"""


# ─────────────────────────────────────────────────────────────
# Parser
# ─────────────────────────────────────────────────────────────

class ResumeParser:
    def __init__(self, api_key=None):
        from google import genai
        self._client = genai.Client(api_key=api_key or os.environ["GEMINI_API_KEY"])

    def _clean_llm_output(self, text: str) -> str:
        """Remove markdown wrappers if present"""
        text = text.strip()
        text = re.sub(r"^```json", "", text)
        text = re.sub(r"```$", "", text)
        return text.strip()

    def _sanitize(self, data: dict, resume_text: str) -> dict:
        """Fix missing / malformed fields from LLM"""

        # Contact
        contact = data.get("contact", {}) or {}
        for key in ["email", "github", "linkedin", "location"]:
            if contact.get(key) is None:
                contact[key] = ""
        data["contact"] = contact

        # Skills / lists
        for key in ["skills", "education", "publications", "awards"]:
            if not isinstance(data.get(key), list):
                data[key] = []

        # Experiences
        for e in data.get("experiences", []):
            e.setdefault("title", "")
            e.setdefault("organization", "")
            e.setdefault("duration", "")
            e.setdefault("bullets", [])
            e.setdefault("technologies", [])
            e.setdefault("metrics", [])

        # Projects
        for p in data.get("projects", []):
            p.setdefault("name", "")
            p.setdefault("description", "")
            p.setdefault("technologies", [])
            p.setdefault("metrics", [])
            p.setdefault("source_sentences", [])

        data["raw_text"] = resume_text
        return data

    @timed("parser.extract")
    def extract(self, resume_text: str) -> FactSheet:
        prompt = EXTRACTION_PROMPT.format(resume_text=resume_text)

        resp = self._client.models.generate_content(
            model=LLM_MODEL_NAME,
            contents=prompt
        )

        raw = self._clean_llm_output(resp.text)

        try:
            data = json.loads(raw)
        except Exception as e:
            raise ValueError(f"LLM returned invalid JSON:\n{raw[:500]}") from e

        data = self._sanitize(data, resume_text)

        return FactSheet(**data)

    def summary(self, fs: FactSheet) -> str:
        n_metrics = sum(len(e.metrics) for e in fs.experiences) + \
                    sum(len(p.metrics) for p in fs.projects)

        return (
            f"Candidate: {fs.candidate_name} | "
            f"Skills: {len(fs.skills)} | "
            f"Experiences: {len(fs.experiences)} | "
            f"Projects: {len(fs.projects)} | "
            f"Metrics: {n_metrics} | "
            f"Publications: {len(fs.publications)}"
        )


print("✅ ResumeParser (robust version) ready")

✅ ResumeParser (robust version) ready


---
## 🏗️ MODULE C — FAISS RAG RETRIEVAL
Exact inner-product search on L2-normalised embeddings = cosine similarity. For corpora >100K, swap to `IndexIVFPQ` for 16× memory reduction.

In [5]:
import faiss

@dataclass
class JobDocument:
    id: str; title: str; company: str; description: str
    level: str = "mid"; domain: str = "ml"

class JobCorpus:
    def __init__(self, ingestion: IngestionLayer, dim=EMBED_DIM):
        self._ing = ingestion
        self._idx = faiss.IndexFlatIP(dim)
        self._docs: list[JobDocument] = []

    def add(self, doc: JobDocument):
        vec = self._ing.embed(doc.description)
        self._idx.add(vec.reshape(1,-1).astype(np.float32))
        self._docs.append(doc)

    def add_batch(self, docs):
        for d in docs: self.add(d)
        console.print(f"✅ Indexed [bold]{len(docs)}[/bold] jobs into FAISS")

    @timed("rag.retrieve")
    def retrieve(self, qvec: np.ndarray, top_k=TOP_K_RETRIEVAL,
                 domain_filter=None) -> list[tuple[JobDocument, float]]:
        fetch_k = top_k * 3 if domain_filter else top_k
        scores, idxs = self._idx.search(qvec.reshape(1,-1).astype(np.float32), fetch_k)
        results = []
        for sc, i in zip(scores[0], idxs[0]):
            if i < 0: continue
            d = self._docs[i]
            if domain_filter and d.domain != domain_filter: continue
            results.append((d, float(sc)))
            if len(results) == top_k: break
        return results

    def __len__(self): return len(self._docs)


print("✅ FAISS JobCorpus defined")


✅ FAISS JobCorpus defined


---
## 🏗️ MODULE D — CONTRASTIVE RANKER + CALIBRATION + BINARY CLASSIFICATION METRICS

**Architecture:** 4-way interaction MLP (u, v, |u-v|, u⊙v) → 1536→512→128→1 with GELU + LayerNorm + Dropout.

**Loss:** MarginRankingLoss — forces `score(resume, positive_job) > score(resume, negative_job) + margin`.
This directly optimises ranking quality rather than independently classifying each pair.

After training the ranker,
1. **Platt scaling calibration** maps raw ranker scores to calibrated probabilities.
2. **Binary classification metrics** (Accuracy, Precision, Recall, F1, AUC-ROC) are computed at a threshold derived from the calibration step.
This makes the ranker score interpretable as a confidence value.

In [6]:
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression


class ResumeJobRanker(nn.Module):
    """
    Pairwise scoring MLP.
    Input: [u; v; |u-v|; u⊙v]  (1536-dim for all-MiniLM-L6-v2)
    Output: scalar in [0,1]
    Dropout kept active at inference for MC uncertainty estimation.
    """
    def __init__(self, emb_dim=EMBED_DIM, dropout_p=0.20):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim*4, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(dropout_p),
            nn.Linear(512, 128),       nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout_p),
            nn.Linear(128, 1), nn.Sigmoid(),
        )

    def forward(self, r, j):
        return self.net(torch.cat([r, j, torch.abs(r-j), r*j], dim=-1)).squeeze(-1)


class PairwiseDataset(Dataset):
    def __init__(self, r, pos, neg):
        self.r   = torch.tensor(r,   dtype=torch.float32)
        self.pos = torch.tensor(pos, dtype=torch.float32)
        self.neg = torch.tensor(neg, dtype=torch.float32)
    def __len__(self): return len(self.r)
    def __getitem__(self, i): return self.r[i], self.pos[i], self.neg[i]


class ContrastiveTrainer:
    """
    Trains ResumeJobRanker with MarginRankingLoss.
    After training: calibrates scores with Platt scaling and reports
    AUC-ROC, F1, Accuracy, Precision, Recall on the eval dataset.
    """
    def __init__(self, emb_dim=EMBED_DIM, margin=0.3, lr=1e-3, epochs=RANKER_EPOCHS):
        self.model   = ResumeJobRanker(emb_dim)
        self.loss_fn = nn.MarginRankingLoss(margin=margin)
        self.optim   = optim.AdamW(self.model.parameters(), lr=lr, weight_decay=1e-4)
        self.epochs  = epochs
        self.history: list[float] = []
        self._calibrator = None   # Platt scaling (LogisticRegression on raw scores)

    def fit(self, dataset: PairwiseDataset, batch_size=16):
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        self.model.train()
        console.print(f"\n[bold]Training contrastive ranker[/bold] — {len(dataset)} pairs, {self.epochs} epochs")
        for epoch in range(1, self.epochs+1):
            epoch_loss = 0.0
            for r, pos, neg in loader:
                self.optim.zero_grad()
                sp = self.model(r, pos); sn = self.model(r, neg)
                loss = self.loss_fn(sp, sn, torch.ones(len(r)))
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optim.step()
                epoch_loss += loss.item()
            avg = epoch_loss / len(loader)
            self.history.append(avg)
            if epoch % 3 == 0:
                console.print(f"  Epoch {epoch:02d}/{self.epochs} — loss {avg:.4f}")
        console.print("[bold green]✅ Ranker trained[/bold green]")

    def calibrate(self, eval_dataset: list[dict], ingestion: IngestionLayer,
                   fit_on: list[dict] = None):
        """
        Platt scaling with separate fit/eval split to prevent data leakage.
        fit_on = training split for fitting the calibrator.
        eval_dataset = held-out eval split for metric reporting.
        Falls back to single-set mode if fit_on is None (backward compat).
        """
        self.model.eval()
        def _get_scores_labels(dataset):
            raw_s, lbls = [], []
            with torch.no_grad():
                for e in dataset:
                    r_emb = ingestion.embed(e["resume"])
                    j_emb = ingestion.embed(e["job"])
                    raw_s.append(self.predict_raw(r_emb, j_emb))
                    lbls.append(e["label"])
            return np.array(raw_s).reshape(-1, 1), np.array(lbls)

        # Fit calibrator on fit_on split (or eval if fit_on not provided)
        fit_set = fit_on if fit_on is not None else eval_dataset
        raw_fit, lbl_fit = _get_scores_labels(fit_set)
        self._calibrator = LogisticRegression(C=1.0, max_iter=1000)
        self._calibrator.fit(raw_fit, lbl_fit)

        # Evaluate on held-out eval_dataset
        raw_arr, lbl_arr = _get_scores_labels(eval_dataset)

        # Binary classification metrics at threshold=0.5
        cal_probs = self._calibrator.predict_proba(raw_arr)[:, 1]
        preds     = (cal_probs >= 0.5).astype(int)

        metrics = {
            "AUC-ROC":   round(roc_auc_score(lbl_arr, cal_probs), 3),
            "Accuracy":  round(accuracy_score(lbl_arr, preds),     3),
            "Precision": round(precision_score(lbl_arr, preds, zero_division=0), 3),
            "Recall":    round(recall_score(lbl_arr, preds, zero_division=0),    3),
            "F1":        round(f1_score(lbl_arr, preds, zero_division=0),        3),
        }
        table = Table(title="📊 Ranker Classification Metrics (post-calibration)", header_style="bold cyan")
        for col in ["Metric", "Value", "Interpretation"]:
            table.add_column(col)
        interp = {
            "AUC-ROC":   "1.0 = perfect ranking",
            "Accuracy":  "Overall correct predictions",
            "Precision": "Of predicted matches, how many are real",
            "Recall":    "Of real matches, how many we found",
            "F1":        "Harmonic mean of P & R",
        }
        for k, v in metrics.items():
            table.add_row(k, str(v), interp[k])
        console.print(table)
        return metrics

    def predict_raw(self, r: np.ndarray, j: np.ndarray) -> float:
        """Uncalibrated score — used internally."""
        self.model.eval()
        with torch.no_grad():
            r_t = torch.tensor(r, dtype=torch.float32).unsqueeze(0)
            j_t = torch.tensor(j, dtype=torch.float32).unsqueeze(0)
            return float(self.model(r_t, j_t).item())

    def predict(self, r: np.ndarray, j: np.ndarray) -> float:
        """Calibrated probability of match. Falls back to raw if calibrator not fitted."""
        raw = self.predict_raw(r, j)
        if self._calibrator is None: return raw
        return float(self._calibrator.predict_proba([[raw]])[0, 1])


# ── MC Dropout Uncertainty ──────────────────────────────────────────────────
@dataclass
class MatchEstimate:
    mean: float; std: float; ci_low: float; ci_high: float

    @property
    def is_uncertain(self): return self.std > 0.10

    def __str__(self):
        flag = "  ⚠ UNCERTAIN" if self.is_uncertain else ""
        return f"mean={self.mean:.3f} ± {self.std:.3f}  95% CI [{self.ci_low:.3f}, {self.ci_high:.3f}]{flag}"


class MCDropoutEstimator:
    """
    Monte Carlo Dropout uncertainty estimation.
    Keeps dropout active at inference → N stochastic forward passes → empirical distribution.
    std > 0.10 → flag as uncertain (insufficient confidence for job recommendation).
    """
    def __init__(self, model: ResumeJobRanker, n_passes=MC_DROPOUT_PASSES):
        self._model = model; self._n = n_passes

    def estimate(self, r: np.ndarray, j: np.ndarray) -> MatchEstimate:
        self._model.train()  # keep dropout active
        scores = []
        with torch.no_grad():
            r_t = torch.tensor(r, dtype=torch.float32).unsqueeze(0)
            j_t = torch.tensor(j, dtype=torch.float32).unsqueeze(0)
            for _ in range(self._n):
                scores.append(float(self._model(r_t, j_t).item()))
        a = np.array(scores)
        return MatchEstimate(float(a.mean()), float(a.std()),
                             float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5)))


print("✅ ContrastiveTrainer + Calibration + MCDropout defined")


✅ ContrastiveTrainer + Calibration + MCDropout defined


---
## 🏗️ MODULE E — LEARNED HALLUCINATION DETECTOR ★ NEW

### Why go beyond rules?
The rule-based validator catches known hallucination patterns (new skill tokens, new numbers, new org names),
but it misses **semantic hallucinations** — cases where the LLM rephrases a fact so far from the source that the
meaning diverges, without introducing any new tokens.

### Design
A binary MLP classifier:
- **Input (4 features):** cosine similarity between bullet and source_fact embeddings,
  plus three asymmetry features that capture one-sided content
- **Output:** P(hallucination) ∈ [0, 1]
- **Training data:** Synthetically generated (bullet, source_fact) pairs with binary labels:
  - **Positive (hallucination = 1):** bullet perturbed by injecting unseen skills/metrics
  - **Negative (grounded = 0):** bullet is a faithful paraphrase of the source

### Why a neural detector vs. just cosine threshold?
A threshold on cosine sim alone cannot distinguish between:
- Low similarity from content divergence (hallucination)
- Low similarity from valid abstraction (not a hallucination)
The learned model sees additional asymmetry features that help separate these cases.

In [7]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report


class LearnedHallucinationDetector:
    """
    Binary hallucination classifier.

    Input features per (bullet, source_fact) pair:
      f1: cosine_sim(bullet_emb, source_emb)          — overall semantic similarity
      f2: max(0, norm(source_emb) - norm(bullet_emb)) — source richer than bullet (abstraction ok)
      f3: L2 distance(bullet_emb, source_emb)         — euclidean drift
      f4: cosine_sim(bullet_emb, mean(source_tokens)) — sentence-level coverage

    Trained on synthetically perturbed (bullet, source) pairs:
      Hallucination = 1: source fact injected with unseen skills/metrics
      Grounded     = 0: faithful paraphrase (word swap, passive→active voice)

    Training is self-supervised — no human labels needed.
    """

    # Pool of synthetic skills/metrics NOT in the eval dataset
    _INJECTED_SKILLS  = [
        "Kubernetes", "Terraform", "Airflow", "Spark", "Kafka", "dbt",
        "TensorRT", "vLLM", "MLflow", "Kubeflow", "BigQuery", "Flink",
        "Snowflake", "Redshift", "Cassandra", "RabbitMQ", "Celery",
        "Prometheus", "Grafana", "Istio", "ArgoCD", "Helm",
        "JAX", "Triton", "TorchServe", "BentoML", "Ray",
    ]
    _INJECTED_METRICS = [
        "99.9%", "10K QPS", "500K req/day", "2M users", "64 GPUs", "20ms p99",
        "3x throughput", "50% cost reduction", "1B parameters", "100ms SLA",
        "99.95% uptime", "12x speedup", "40% latency reduction", "1M events/sec",
    ]

    # Manually curated validation set (15 examples with known ground-truth)
    CURATED_VAL_SET = [
        # (bullet, source_fact, label)  0=grounded, 1=hallucinated
        ("Designed DenseNet201 pipeline for multi-label X-ray classification with 0.92 recall",
         "Designed DenseNet201 pipeline for multi-label chest X-ray classification, 14 pathologies, 0.92 recall, 0.86 specificity",
         0),
        ("Built hierarchical RAG system reducing irrelevant context by 40% vs flat retrieval",
         "Two-stage hierarchical retrieval over 100+ PDFs; reduced irrelevant context by 40% vs flat retrieval baseline",
         0),
        ("Deployed Flask REST APIs on NVIDIA Quadro GPU systems with result caching",
         "Deployed Flask REST APIs on NVIDIA Quadro multi-GPU systems with result caching",
         0),
        ("Implemented Kafka streaming pipeline achieving 1M events/sec throughput",
         "Deployed Flask REST APIs on NVIDIA Quadro multi-GPU systems with result caching",
         1),
        ("Used Kubernetes and Terraform to deploy ML models at 99.9% uptime",
         "Designed DenseNet201 pipeline for multi-label chest X-ray classification, 14 pathologies, 0.92 recall",
         1),
        ("Built LangGraph-based agentic pipeline with PubMed API and FAISS indexing",
         "LangGraph-based agentic pipeline: PubMed API search, FAISS indexing, semantic retrieval",
         0),
        ("Implemented anatomical zone segmentation with GradCAM saliency across six lung regions",
         "Implemented anatomical zone segmentation (CXAS model) across six lung regions with GradCAM saliency maps",
         0),
        ("Achieved sub-100ms retrieval latency using Snowflake and Redshift on 10B-row corpus",
         "Sub-100ms retrieval latency on single CPU node; BAAI/bge-large-en embeddings",
         1),
        ("Addressed class imbalance using focal loss and weighted binary cross-entropy",
         "Addressed class imbalance using focal loss and weighted binary cross-entropy",
         0),
        ("Published at IEEE CVMI 2025 on Data Poisoning Attacks and Defenses in Machine Learning",
         "IEEE CVMI 2025: Data Poisoning Attacks and Defenses in Machine Learning",
         0),
        ("Scaled to 64 GPUs using Ray and TorchServe for 3x throughput improvement",
         "Field-deployed in National Health Mission Chennai mobile TB screening ambulances",
         1),
        ("Recognized among 196 global TB innovations by the Gates Foundation",
         "XrayCAD selected among 196 global TB innovations (Gates Foundation / UNION)",
         0),
        ("Built RAG system with Cassandra backend serving 2M users at <5ms p99 latency",
         "Two-stage hierarchical retrieval over 100+ PDFs; reduced irrelevant context by 40%",
         1),
        ("nomic-embed-text embeddings with citation-grounded prompting in clinical LLM agent",
         "nomic-embed-text embeddings; citation-grounded prompting; fault-tolerant tool dispatcher",
         0),
        ("M.Tech Data Science, DIAT Pune, 2024, CGPA 8.16",
         "M.Tech, Data Science, DIAT Pune, 2024, CGPA 8.16",
         0),
    ]

    def __init__(self, ingestion: IngestionLayer):
        self._ing   = ingestion
        self._model = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(64, 32),   # slightly wider
                activation="relu",
                max_iter=HALLUC_EPOCHS,
                random_state=42,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=10,           # more patience
            )),
        ])
        self._fitted = False

    def _features(self, bullet: str, source_fact: str) -> np.ndarray:
        """Extract the 4-dimensional feature vector for a (bullet, source_fact) pair."""
        b_emb = self._ing.embed(bullet)
        s_emb = self._ing.embed(source_fact)
        cos   = float(np.dot(b_emb, s_emb))                  # f1: cosine similarity
        asymm = max(0.0, float(np.linalg.norm(s_emb)) -
                         float(np.linalg.norm(b_emb)))        # f2: source richer
        l2    = float(np.linalg.norm(b_emb - s_emb))         # f3: L2 drift
        # f4: sentence-level coverage (how many source sentences are covered by bullet)
        sents, svecs = self._ing.embed_sentences(source_fact)
        sent_sims    = [float(np.dot(b_emb, sv)) for sv in svecs]
        coverage     = float(np.mean(sent_sims)) if sent_sims else cos
        return np.array([cos, asymm, l2, coverage])

    def _generate_training_data(self, source_facts: list[str]) -> tuple[np.ndarray, np.ndarray]:
        X, y = [], []
        random.seed(42)
        n = len(source_facts)
    
        for i, sf in enumerate(source_facts):
            words = sf.split()
    
            # ── GROUNDED: 3 paraphrases per source (was 1) ──────────────────
            for drop_rate in [0.10, 0.15, 0.20]:
                paraphrase = " ".join(
                    w if random.random() > drop_rate else (w[::-1] if len(w) > 4 else w)
                    for w in words
                )
                X.append(self._features(paraphrase, sf)); y.append(0)
    
            # Also add the source fact itself as a grounded example
            X.append(self._features(sf, sf)); y.append(0)
    
            # ── HALLUCINATED: 3 variants per source (was 1) ──────────────────
            for offset in [n // 3, n // 2, (n * 2) // 3]:
                other_sf = source_facts[(i + offset) % n]
                injected_skill  = random.choice(self._INJECTED_SKILLS)
                injected_metric = random.choice(self._INJECTED_METRICS)
                other_sent = other_sf.split(".")[0].strip()
                halluc = f"{other_sent}. Scaled with {injected_skill} to {injected_metric}."
                X.append(self._features(halluc, sf)); y.append(1)
    
        return np.array(X), np.array(y)

    def train(self, source_facts: list[str]) -> dict:
        """
        Train on expanded synthetic data, then evaluate on curated validation set.
        Reports Precision/Recall/F1 on both synthetic and real curated examples.
        """
        from sklearn.model_selection import train_test_split
        X, y = self._generate_training_data(source_facts)
        console.print(f"\n[bold]Training Hallucination Detector [/bold] — {len(X)} synthetic pairs")
        console.print(f"  Features: cosine_sim, source_asymmetry, L2_drift, coverage")

        # Proper train/test split on synthetic data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )
        
        # Train model
        self._model.fit(X_train, y_train)
        self._fitted = True
        
        # 🔥 Threshold calibration (ADD HERE)
        train_proba = self._model.predict_proba(X_train)[:, 1]
        
        from sklearn.metrics import roc_curve
        fpr, tpr, thresholds = roc_curve(y_train, train_proba)
        
        
        viable = [(t, tp) for t, tp in zip(thresholds, tpr) if tp >= 0.6]
        self._threshold = viable[0][0] if viable else 0.35
        
        # ──targets precision >= 0.85 → only reject when confident
        from sklearn.metrics import precision_recall_curve
        precisions, recalls, pr_thresholds = precision_recall_curve(y_train, train_proba)
        # Find lowest threshold where precision >= 0.85
        viable = [
            (t, p, r) for t, p, r in zip(pr_thresholds, precisions[:-1], recalls[:-1])
            if p >= 0.85
        ]
        if viable:
            # Pick the one with highest recall among those meeting precision target
            self._threshold = max(viable, key=lambda x: x[2])[0]
        else:
            self._threshold = 0.60  # safe fallback — don't fire unless fairly confident
        console.print(f"  [dim]Auto-calibrated threshold (precision≥0.85): {self._threshold:.3f}[/dim]")

        # Evaluate on synthetic held-out test set
        proba_test = self._model.predict_proba(X_test)[:, 1]
        preds_test = (proba_test >= self._threshold).astype(int)
        report_synth = classification_report(y_test, preds_test, target_names=["grounded","hallucinated"], output_dict=True)

        table = Table(title="Hallucination Detector — Synthetic Test Set Metrics", header_style="bold magenta")
        table.add_column("Class"); table.add_column("Precision"); table.add_column("Recall"); table.add_column("F1"); table.add_column("Support")
        for cls in ["grounded", "hallucinated"]:
            r = report_synth[cls]
            table.add_row(cls, f"{r['precision']:.3f}", f"{r['recall']:.3f}", f"{r['f1-score']:.3f}", str(int(r['support'])))
        console.print(table)

        # Evaluate on curated validation set (manually labelled, 15 examples)
        X_cur = np.array([self._features(b, s) for b, s, _ in self.CURATED_VAL_SET])
        y_cur = np.array([lbl for _, _, lbl in self.CURATED_VAL_SET])
        proba_cur = self._model.predict_proba(X_cur)[:, 1]
        preds_cur = (proba_cur >= self._threshold).astype(int)
        proba_cur = self._model.predict_proba(X_cur)[:, 1]
        report_cur = classification_report(y_cur, preds_cur, target_names=["grounded","hallucinated"], output_dict=True)
        from sklearn.metrics import roc_auc_score as _auc
        auc_cur = _auc(y_cur, proba_cur) if len(set(y_cur)) > 1 else 0.0

        table2 = Table(title=f"Hallucination Detector — Curated Val Set (n={len(y_cur)}, AUC={auc_cur:.3f})", header_style="bold yellow")
        table2.add_column("Class"); table2.add_column("Precision"); table2.add_column("Recall"); table2.add_column("F1")
        for cls in ["grounded", "hallucinated"]:
            r = report_cur[cls]
            table2.add_row(cls, f"{r['precision']:.3f}", f"{r['recall']:.3f}", f"{r['f1-score']:.3f}")
        console.print(table2)
        console.print(f"  [bold]Curated set overall accuracy: {report_cur['accuracy']:.3f} | AUC: {auc_cur:.3f}[/bold]")

        return {"synthetic": report_synth, "curated": report_cur, "curated_auc": auc_cur}

    def predict_proba(self, bullet: str, source_fact: str) -> float:
        """Returns P(hallucination) in [0, 1]."""
        if not self._fitted:
            raise RuntimeError("Call .train() before .predict_proba()")
        feats = self._features(bullet, source_fact).reshape(1, -1)
        return float(self._model.predict_proba(feats)[0, 1])

    def is_hallucination(self, bullet: str, source_fact: str,
                          threshold=CONFIDENCE_THRESHOLD) -> tuple[bool, float]:
        """Returns (is_hallucination, p_halluc)."""
        p = self.predict_proba(bullet, source_fact)
        return p > threshold, p


print("✅ LearnedHallucinationDetector defined")


✅ LearnedHallucinationDetector defined


---
## 🏗️ MODULE F — CONTROLLED OPTIMIZER (Stage 2)
Takes ONLY the FactSheet JSON. Never sees raw resume text.

In [8]:
OPTIMIZE_PROMPT = """You are a factual resume optimizer operating under STRICT GROUNDING constraints.

ABSOLUTE CONSTRAINTS — enforced by a downstream ML hallucination classifier:
1. ONLY use skills, experiences, projects, metrics explicitly stated in the FACT SHEET.
2. Do NOT add new skills, tools, organizations, or numbers absent from the FACT SHEET.
3. Do NOT infer, generalise, or abstract beyond the original source sentence.
4. Every bullet MUST reuse key keywords directly from the source_fact it references.
5. STAY as close as possible to the exact wording of source_fact — rephrase minimally.
6. "Do not generalize or abstract beyond the original sentence. Stay as close as possible to the source_fact."

FACT SHEET:
{factsheet_json}

JOB DESCRIPTION:
{job_description}

Return JSON:
{{
  "summary": "2-sentence summary using ONLY facts from FACT SHEET",
  "bullets": [
    {{
      "bullet": "STAR-format bullet — must be semantically equivalent to source_fact, reusing its keywords",
      "source_fact": "exact sentence/metric copied verbatim from FACT SHEET"
    }}
  ],
  "cover_letter": "3-paragraph cover letter. Facts only. No invented achievements."
}}

Rules: max 8 bullets, strong action verbs, real metrics only, mirror JD language.
Return ONLY valid JSON. No markdown fences."""

# Stricter prompt for regeneration retries (escalating constraints)
REGEN_PROMPTS = [
    # Attempt 1 — stricter
    """You are a factual resume optimizer. A bullet you generated was REJECTED as hallucinated.
STRICT REGENERATION — attempt 1.

RULES (NON-NEGOTIABLE):
1. The new bullet MUST be semantically equivalent to the SOURCE FACT below.
2. You MUST reuse at least 3 keywords verbatim from SOURCE FACT.
3. Do NOT introduce any new skills, tools, metrics, or organisations.
4. Do NOT abstract or generalise beyond the SOURCE FACT.

SOURCE FACT: {source_fact}
JOB DESCRIPTION (for style guidance only): {jd_snippet}
REJECTED BULLET: {rejected_bullet}

Return JSON ONLY: {{"bullet": "...", "source_fact": "{source_fact}"}}""",

    # Attempt 2 — near-verbatim
    """FINAL REGENERATION — attempt 2. Previous attempts were rejected.

You must produce a bullet that is almost a direct copy of the SOURCE FACT, lightly reformatted for resume style.
Reuse EVERY key metric, skill, and entity from the SOURCE FACT. Change only verb tense and sentence structure.

SOURCE FACT: {source_fact}
REJECTED BULLET: {rejected_bullet}

Return JSON ONLY: {{"bullet": "...", "source_fact": "{source_fact}"}}""",

    # Attempt 3 — verbatim fallback
    """FINAL FALLBACK — attempt 3.
Return the SOURCE FACT below converted directly into a resume bullet with zero creative changes.
Do NOT omit, add, or paraphrase any fact. Only add a strong action verb at the start if absent.

SOURCE FACT: {source_fact}

Return JSON ONLY: {{"bullet": "...", "source_fact": "{source_fact}"}}""",
]

@dataclass
class EvidencedBullet:
    bullet: str
    source_fact: str
    p_hallucination: float = 0.0   # filled by hallucination detector
    rejected: bool = False
    regen_attempts: int = 0        


@dataclass
class OptimizedResume:
    candidate_name: str
    target_role: str
    summary: str
    evidenced_bullets: list[EvidencedBullet]
    cover_letter: str
    factsheet_snapshot: dict

    @property
    def summary_stats(self) -> str:
        total = len(self.evidenced_bullets)
        regens = sum(1 for e in self.evidenced_bullets if e.regen_attempts > 0)
        return f"{total} bullets ({regens} regenerated)"


class ControlledOptimizer:
    def __init__(self, api_key=None):
        from google import genai
        self._client = genai.Client(api_key=api_key or os.environ["GEMINI_API_KEY"])

    def _fs_json(self, fs: "FactSheet") -> str:
        d = fs.model_dump(); d.pop("raw_text", None)
        return json.dumps(d, indent=2)

    @timed("optimizer.generate")
    def optimize(self, factsheet: "FactSheet", job_description: str) -> OptimizedResume:
        prompt = OPTIMIZE_PROMPT.format(
            factsheet_json=self._fs_json(factsheet),
            job_description=job_description[:3000],
        )
        resp = self._client.models.generate_content(
            model=LLM_MODEL_NAME, contents=prompt,
            config={"temperature": 0.0},
        )
        raw  = re.sub(r'^```json\s*', '', resp.text.strip())
        raw  = re.sub(r'```\s*$', '', raw)
        data = json.loads(raw)
        bullets = [EvidencedBullet(b["bullet"], b["source_fact"]) for b in data.get("bullets", [])]
        snap = {
            "skills": factsheet.skills,
            "metrics": [m.value for e in factsheet.experiences for m in e.metrics] +
                       [m.value for p in factsheet.projects for m in p.metrics],
            "organizations": [e.organization for e in factsheet.experiences] +
                             ([factsheet.education[0]["institution"]] if factsheet.education else []),
        }
        return OptimizedResume(
            candidate_name=factsheet.candidate_name,
            target_role=job_description[:60],
            summary=data.get("summary",""),
            evidenced_bullets=bullets,
            cover_letter=data.get("cover_letter",""),
            factsheet_snapshot=snap,
        )

    @timed("optimizer.regenerate")
    def regenerate_bullet(self, rejected_bullet: str, source_fact: str,
                          job_description: str, attempt: int) -> EvidencedBullet:
        """Regenerate a single failed bullet with escalating strictness."""
        prompt_template = REGEN_PROMPTS[min(attempt, len(REGEN_PROMPTS)-1)]
        prompt = prompt_template.format(
            source_fact=source_fact,
            rejected_bullet=rejected_bullet,
            jd_snippet=job_description[:500],
        )
        try:
            resp = self._client.models.generate_content(
                model=LLM_MODEL_NAME, contents=prompt,
                config={"temperature": 0.0},
            )
            raw  = re.sub(r'^```json\s*', '', resp.text.strip())
            raw  = re.sub(r'```\s*$', '', raw)
            data = json.loads(raw)
            return EvidencedBullet(
                bullet=data.get("bullet", source_fact),
                source_fact=source_fact,
                regen_attempts=attempt+1,
            )
        except Exception:
            # Fallback: use source_fact directly as the bullet
            return EvidencedBullet(
                bullet=source_fact,
                source_fact=source_fact,
                regen_attempts=attempt+1,
            )


print("✅ ControlledOptimizer (with regeneration loop) defined")


✅ ControlledOptimizer (with regeneration loop) defined


---
## 🏗️ MODULE G — CONFIDENCE GATE + RULE-BASED FALLBACK VALIDATOR

**Confidence gate:** After the learned detector scores each bullet, bullets with `P(halluc) > CONFIDENCE_THRESHOLD`
are **rejected** and the system can request a regeneration with a stricter prompt.

**Rule-based fallback:** The regex checks (skill/metric/org drift) run as a secondary layer.
If either the learned detector OR the rule checker flags a bullet, it is marked for review.
This two-layer approach gives defence-in-depth.

In [9]:
@dataclass
class HallucinationReport:
    bullet: str
    source_fact: str
    p_hallucination: float      # from learned detector
    is_learned_halluc: bool
    rule_flags: list[str]       # from rule-based fallback
    rule_halluc_type: str       # 'skill'|'metric'|'org'|'none'
    faithfulness_cosine: float  # cosine(bullet_emb, source_emb)
    final_verdict: bool         # True = hallucination (either detector says so)

    @property
    def confidence(self) -> float:
        return 1.0 - self.p_hallucination


GENERIC_TERMS = {
    "python","api","rest","ml","ai","llm","nlp","cv","sql","json","git","linux","docker",
    "cloud","deep","learning","model","data","system","pipeline","production","engineer",
    "trained","deployed","built","designed","implemented","agentic","retrieval","generation",
}


class RuleBasedFallback:
    """Regex / token-matching checks kept as secondary layer after learned detector."""

    def _normalize(self, s): return re.sub(r'[^a-z0-9]', '', s.lower())
    def _nums(self, t): return set(re.findall(r'\b\d+\.?\d*%?\b', t))

    def check(self, bullet: str, snapshot: dict) -> tuple[list[str], str]:
        known_skills_n = {self._normalize(s) for s in snapshot.get("skills", [])}
        known_orgs_n   = {self._normalize(o) for o in snapshot.get("organizations", [])}
        known_nums     = set()
        for m in snapshot.get("metrics", []):
            known_nums.update(self._nums(str(m)))

        flagged = []; htype = "none"

        # Skill drift
        for tok in set(re.findall(r'\b[A-Za-z][A-Za-z0-9_\-.]{2,}\b', bullet)):
            n = self._normalize(tok)
            if n in GENERIC_TERMS: continue
            if (tok[0].isupper() or '-' in tok) and n not in known_skills_n:
                if not any(n in k or k in n for k in known_skills_n):
                    flagged.append(tok); htype = "skill"

        # Metric drift
        new_nums = self._nums(bullet) - known_nums
        if new_nums:
            flagged.extend(list(new_nums))
            htype = htype if htype != "none" else "metric"

        # Org drift
        for phrase in re.findall(r'\b[A-Z][a-zA-Z0-9]*(?:\s+[A-Z][a-zA-Z0-9]*)+\b', bullet):
            pn = self._normalize(phrase)
            if pn not in known_orgs_n and pn not in known_skills_n:
                if not any(pn in k or k in pn for k in known_orgs_n | known_skills_n):
                    flagged.append(phrase); htype = htype if htype != "none" else "org"

        return flagged, htype


class ConfidenceGatedValidator:
    """
    Two-layer hallucination audit:
    Layer 1: LearnedHallucinationDetector (ML-based, semantic)
    Layer 2: RuleBasedFallback (regex, token matching)
    Final verdict = OR of both layers.
    Bullets with P(halluc) > CONFIDENCE_THRESHOLD are marked for rejection/regeneration.
    """

    def __init__(self, detector: LearnedHallucinationDetector, ingestion: IngestionLayer):
        self._det    = detector
        self._rules  = RuleBasedFallback()
        self._ing    = ingestion

    def validate_bullet(self, eb: EvidencedBullet, snapshot: dict) -> HallucinationReport:
        # Layer 1: learned
        p_halluc, is_learned = 0.0, False
        if self._det._fitted:
            is_halluc, p_halluc = self._det.is_hallucination(eb.bullet, eb.source_fact)
            is_learned = is_halluc

        # Layer 2: rules
        rule_flags, rule_type = self._rules.check(eb.bullet, snapshot)

        # Faithfulness cosine
        b_emb = self._ing.embed(eb.bullet)
        s_emb = self._ing.embed(eb.source_fact)
        faith = float(np.dot(b_emb, s_emb))

        # Tightened final verdict — stricter combined rejection rule
        # Reject if: learned detector fires (p > 0.22) OR
        #            cosine faithfulness < 0.75 OR
        #            rule flags with cosine < 0.75
        final = (
            is_learned or
            faith < COSINE_FAITHFULNESS_THRESHOLD or
            (bool(rule_flags) and faith < 0.75)
        )

        return HallucinationReport(
            bullet=eb.bullet, source_fact=eb.source_fact,
            p_hallucination=round(p_halluc, 4), is_learned_halluc=is_learned,
            rule_flags=rule_flags, rule_halluc_type=rule_type,
            faithfulness_cosine=round(faith, 4), final_verdict=final,
        )

    def validate_and_gate(self, opt_resume: OptimizedResume,
                          threshold=CONFIDENCE_THRESHOLD) -> tuple[list[HallucinationReport], OptimizedResume]:
        """
        FIX: Validate all bullets with tightened thresholds.
        Note: regeneration loop is handled in Cell 14 (run_regeneration_loop).
        This method validates after each regeneration attempt.
        Returns (reports, updated_opt_resume).
        """
        reports = []
        for eb in opt_resume.evidenced_bullets:
            r = self.validate_bullet(eb, opt_resume.factsheet_snapshot)
            eb.p_hallucination = r.p_hallucination
            eb.rejected        = r.final_verdict
            reports.append(r)
        return reports, opt_resume

    def hallucination_rate(self, reports): return sum(r.final_verdict for r in reports) / len(reports) if reports else 0.0
    def avg_faithfulness(self, reports):   return sum(r.faithfulness_cosine for r in reports) / len(reports) if reports else 0.0


print("✅ ConfidenceGatedValidator defined")


✅ ConfidenceGatedValidator defined


---
## 🏗️ MODULE H — GRADIENT ATTRIBUTION EXPLAINER
**Why does this job match this resume?**

Gradient × Input attribution maps importance back from the ranker's output to individual resume/JD sentences.
This satisfies "explainability" without needing the full SHAP background dataset.

In [10]:
class GradientExplainer:
    """
    Gradient × Input attribution on the contrastive ranker.

    For a (resume, job) pair:
    1. Forward pass → score.
    2. Backprop → grad w.r.t. resume embedding and job embedding.
    3. attr_i = |grad_i × emb_i| / sum  — normalised importance weights per dim.
    4. Top-3 sentences ranked by attribution-weighted activation:
       score(sentence) = sentence_embedding · attr_weights
       This asks: "which sentence activates the dimensions the ranker cares about?"

    Note: embed_sentences() pre-filters education/publication/award lines so
    they cannot appear here regardless of their gradient score.

    Output: {'resume_drivers': [(sentence, importance)], 'job_drivers': [...]}
    """

    def __init__(self, model: ResumeJobRanker, ingestion: IngestionLayer):
        self._model = model; self._ing = ingestion

    def _attr(self, r_emb, j_emb, is_resume: bool) -> np.ndarray:
        """
        Returns per-dimension importance weights (sum to 1).
        Normalising to importance weights (not L2 norm) keeps the dot product
        in _top_sentences interpretable as a weighted activation score.
        """
        self._model.eval()
        r_t = torch.tensor(r_emb, dtype=torch.float32).unsqueeze(0).requires_grad_(True)
        j_t = torch.tensor(j_emb, dtype=torch.float32).unsqueeze(0).requires_grad_(True)
        score = self._model(r_t, j_t)
        score.backward()
        grad = (r_t if is_resume else j_t).grad.detach().numpy()[0]
        target = r_emb if is_resume else j_emb
        raw = np.abs(grad * target)          # gradient × input, element-wise
        return raw / (raw.sum() + 1e-8)      # normalise → importance weights

    def _top_sentences(self, text: str, attr: np.ndarray, top_k: int = 3,
                       jd_emb: np.ndarray = None, jd_weight: float = 0.3):
        """
        Rank sentences by attribution-weighted activation.

        attr[d] = importance weight of embedding dimension d.
        np.dot(sentence_vec, attr) = how strongly the sentence activates
        the dimensions the ranker found important for this match.

        Optional secondary signal: cosine similarity with the job description
        embedding (jd_emb) blended in to further surface JD-relevant sentences.
        """
        sents, svecs = self._ing.embed_sentences(text)
        if not sents:
            return []

        scored = []
        for s, sv in zip(sents, svecs):
            attr_score = float(np.dot(sv, attr))          # primary: attribution signal
            if jd_emb is not None:
                jd_score = float(np.dot(sv, jd_emb))      # secondary: JD relevance
                final = (1 - jd_weight) * attr_score + jd_weight * jd_score
            else:
                final = attr_score
            scored.append((s, final))

        return sorted(scored, key=lambda x: x[1], reverse=True)[:top_k]

    @timed("explainer.attribute")
    def explain(self, resume_text: str, job_text: str, top_k: int = 3) -> dict:
        r_emb = self._ing.embed(resume_text)
        j_emb = self._ing.embed(job_text)
        return {
            # Resume drivers: use JD embedding as secondary signal so results
            # are both attribution-grounded and JD-relevant.
            "resume_drivers": self._top_sentences(
                resume_text,
                self._attr(r_emb, j_emb, True),
                top_k,
                jd_emb=j_emb,
                jd_weight=0.3,
            ),
            # Job drivers: use resume embedding as secondary signal.
            "job_drivers": self._top_sentences(
                job_text,
                self._attr(r_emb, j_emb, False),
                top_k,
                jd_emb=r_emb,
                jd_weight=0.3,
            ),
        }


print("✅ GradientExplainer defined")


✅ GradientExplainer defined


---
## 🏗️ MODULE I — ENSEMBLE SCORER
Cosine (30%) + Calibrated Ranker (55%) + RAG Boost (15%) with MC Dropout CI.

In [11]:
class EnsembleScorer:
    def __init__(self, ingestion, corpus, trainer, w=(0.30, 0.55, 0.15)):
        self._ing     = ingestion; self._corpus = corpus
        self._trainer = trainer;  self._w = w
        self._mc      = MCDropoutEstimator(trainer.model)
        self._expl    = GradientExplainer(trainer.model, ingestion)

    # ── Hard block: these terms must never appear in gradient drivers ─────────
    _DRIVER_BLOCK = {
        "education", "cgpa", "gpa", "university", "institute", "college",
        "award", "awards", "conference", "publication", "publications",
        "m.tech", "b.tech", "b.e.", "m.e.", "ph.d", "phd",
        "ieee", "icmr", "nirt", "iit", "nit", "bits",
        "research dissemination", "gates foundation",
    }

    def _is_skill_sentence(self, s: str, job_text: str = "") -> bool:
        """
        Returns True if the sentence looks like a real skill/experience sentence
        that belongs in the gradient driver output.

        Three layers of filtering:
        1. Hard block on education / publication / award terms.
        2. Structural filters (length, alpha ratio, contact noise).
        3. JD overlap: must share at least one meaningful word with the JD.
        """
        import re
        s_lower = s.lower()
        s = s.strip()

        # Layer 1: hard block
        if any(term in s_lower for term in self._DRIVER_BLOCK):
            return False
        if re.search(r'\b(cgpa|gpa)\s*[:\-]?\s*[\d.]+', s_lower):
            return False

        # Layer 2: structural filters
        if len(s) < 50:
            return False
        if re.match(r'^[\d\s\./()%\-]+$', s):
            return False
        if len(re.findall(r'\b[a-zA-Z]{3,}\b', s)) < 6:
            return False
        if any(x in s_lower for x in ["github", "linkedin", "http", "www", "@"]):
            return False

        # Layer 3: must overlap with the job description on a meaningful word
        if job_text:
            jd_words = {w for w in job_text.lower().split() if len(w) > 4}
            resume_words = set(s_lower.split())
            if not (jd_words & resume_words):
                return False

        return True

    def _rerank(self, drivers, query_emb, query_text: str) -> list[str]:
        """
        Re-rank drivers by a blended score:
          50% attribution/gradient score (from GradientExplainer)
          40% cosine similarity to query embedding
          10% keyword boost (RAG / ML stack terms)
           2% per matched JD word

        Hard block: any sentence containing a block term is sent to -999
        unconditionally — it cannot win regardless of its gradient score.
        """
        import re
        ranked = []
        jd_words = set(query_text.lower().split())

        ML_KEYWORDS = [
            "pytorch", "faiss", "model", "training", "deployment",
            "inference", "pipeline", "embedding", "retrieval", "agent",
            "rag", "llm", "langchain", "langgraph", "vector", "classification",
            "segmentation", "detection", "transformer", "fine-tun",
        ]

        for item in drivers:
            s, grad_score = item if isinstance(item, tuple) else (item, 1.0)
            s_lower = s.lower()

            # Hard block — cannot surface under any circumstances
            if any(term in s_lower for term in self._DRIVER_BLOCK):
                ranked.append((-999.0, s))
                continue

            emb = self._ing.embed(s)
            sim = float(np.dot(emb, query_emb))
            kw_boost = sum(1 for k in ML_KEYWORDS if k in s_lower)
            jd_match = sum(1 for w in s_lower.split() if w in jd_words and len(w) > 3)

            final_score = (
                0.50 * grad_score +
                0.40 * sim +
                0.05 * min(kw_boost, 4) +   # cap so one keyword-rich sentence can't dominate
                0.02 * min(jd_match, 10)
            )
            ranked.append((final_score, s))

        ranked.sort(reverse=True)
        return [s for _, s in ranked]

    @timed("scorer.score")
    def score(self, resume_text: str, job_text: str,
              with_uncertainty=True, with_explanation=True) -> dict:
        r_emb = self._ing.embed(resume_text)
        j_emb = self._ing.embed(job_text)
        cosine = float(np.dot(r_emb, j_emb))
        ranker = self._trainer.predict(r_emb, j_emb)
        retrieved = self._corpus.retrieve(r_emb)
        rag_boost = 1.0 if job_text[:100] in {d.description[:100] for d,_ in retrieved} else 0.0
        w1, w2, w3 = self._w
        composite  = w1*cosine + w2*ranker + w3*rag_boost
        result = {
            "cosine":    round(cosine*100, 1),
            "ranker":    round(ranker*100, 1),
            "rag_boost": rag_boost,
            "composite": round(composite*100, 1),
        }

        if with_uncertainty:
            est = self._mc.estimate(r_emb, j_emb)
            result["uncertainty"] = str(est)
            result["is_uncertain"] = est.is_uncertain

        if with_explanation:
            expl = self._expl.explain(resume_text, job_text)

            # Filter with JD context so Layer 3 (JD overlap) runs correctly
            clean_resume = [
                (s, sc) for s, sc in expl["resume_drivers"]
                if self._is_skill_sentence(s, job_text)
            ]
            clean_job = [
                (s, sc) for s, sc in expl["job_drivers"]
                if self._is_skill_sentence(s, resume_text)
            ]

            # Safe fallback: if everything was filtered, use length-only filter.
            # The hard block in _rerank still prevents any bad sentence from winning.
            if not clean_resume:
                clean_resume = [(s, sc) for s, sc in expl["resume_drivers"] if len(s) > 50]
            if not clean_job:
                clean_job = [(s, sc) for s, sc in expl["job_drivers"] if len(s) > 50]

            result["resume_drivers"] = self._rerank(clean_resume, j_emb, job_text)[:3]
            result["job_drivers"]    = self._rerank(clean_job,    r_emb, resume_text)[:3]

        return result


print("✅ EnsembleScorer defined")


✅ EnsembleScorer defined


---
## 🏗️ MODULE J — FULL EVALUATION FRAMEWORK
Grounding metrics + IR metrics + ranker classification metrics in one place.

In [12]:
import pandas as pd

@dataclass
class EvalResult:
    hallucination_rate: float
    faithfulness_score: float
    semantic_similarity: float
    jd_relevance: float
    total_bullets: int
    flagged_bullets: int

    def display(self, title="Evaluation"):
        table = Table(title=title, header_style="bold magenta")
        table.add_column("Metric"); table.add_column("Value"); table.add_column("Target")
        rows = [
            ("Hallucination Rate", f"{self.hallucination_rate:.1%}", "< 5%",
             self.hallucination_rate < 0.05),
            ("Faithfulness Score", f"{self.faithfulness_score:.3f}", "> 0.65",
             self.faithfulness_score > 0.65),
            ("Semantic Similarity", f"{self.semantic_similarity:.3f}", "> 0.70",
             self.semantic_similarity > 0.70),
            ("JD Relevance", f"{self.jd_relevance:.3f}", "> 0.55",
             self.jd_relevance > 0.55),
            (f"Flagged Bullets", f"{self.flagged_bullets}/{self.total_bullets}", "0", self.flagged_bullets==0),
        ]
        for name, val, tgt, ok in rows:
            style = "green" if ok else ("red" if name=="Hallucination Rate" else "yellow")
            table.add_row(name, val, tgt, style=style)
        console.print(table)


def precision_at_k(labels, k): return sum(labels[:k])/k
def recall_at_k(labels, k, n): return sum(labels[:k])/n if n else 0.0
def topk_accuracy(labels, k):  return float(any(labels[:k]))
def ndcg_at_k(labels, k):
    def dcg(ls): return sum(l/math.log2(i+2) for i,l in enumerate(ls[:k]))
    ideal = dcg(sorted(labels, reverse=True)); return dcg(labels)/ideal if ideal else 0.0


EVAL_DATASET = [
    # Strong positives — clear matches
    {"id":"E01","label":1,"resume":"Senior ML Engineer. PyTorch, distributed training 32 GPUs, MLflow, FAISS, recommendation systems. Led A/B tests improving CTR 18%. Deployed two-tower retrieval at 20ms p99.","job":"ML Engineer — Recommendations. PyTorch, distributed training, MLOps, A/B testing, two-tower architecture, ranking systems."},
    {"id":"E02","label":1,"resume":"Research engineer, NLP. BERT fine-tuning, RLHF, Hugging Face. 7B param model training on 64-GPU cluster. Published EMNLP paper.","job":"AI Research Engineer. LLM fine-tuning, RLHF, distributed training, NLP, transformer architectures."},
    {"id":"E03","label":1,"resume":"Data scientist. Scikit-learn, XGBoost, Spark feature engineering, W&B experiment tracking. Deployed REST API serving 500K req/day.","job":"ML Engineer. Feature engineering, experiment tracking, model deployment, Python, Spark."},
    {"id":"E04","label":1,"resume":"MLOps engineer. Kubernetes, Kubeflow, model versioning, drift detection, SLO monitoring. 99.9% uptime on 40 production models.","job":"MLOps Engineer. Model serving, Kubernetes, monitoring, drift detection, CI/CD for ML."},
    {"id":"E05","label":1,"resume":"Platform engineer. Built two-tower embedding model, FAISS indexing, real-time feature serving <5ms. Ran 50+ A/B experiments.","job":"ML Platform Engineer. Embedding models, vector search, feature store, real-time serving."},
    {"id":"E06","label":1,"resume":"Computer vision engineer. PyTorch, YOLO, image segmentation, edge deployment on NVIDIA Jetson. 94% mAP on custom dataset.","job":"Computer Vision Engineer. Object detection, segmentation, PyTorch, edge deployment, model optimization."},
    {"id":"E07","label":1,"resume":"LLM infrastructure engineer. vLLM serving, speculative decoding, KV cache optimization. Reduced inference cost 40% at 10K QPS.","job":"AI Infra Engineer. LLM serving, vLLM, inference optimization, GPU utilization, high-throughput systems."},
    {"id":"E08","label":1,"resume":"Data engineer. Apache Kafka, Spark streaming, dbt, BigQuery. Built real-time feature pipeline for 1M events/sec.","job":"Data Engineer. Kafka, Spark, BigQuery, real-time pipelines, feature engineering for ML."},
    {"id":"E09","label":1,"resume":"Backend engineer. Python, FastAPI, PostgreSQL, Redis. Some ML model integration. Docker, Kubernetes.","job":"ML Engineer. Python, model deployment, API development, Docker, Kubernetes."},
    {"id":"E10","label":1,"resume":"Software engineer with ML coursework. PyTorch tutorials, scikit-learn projects. 2 years SWE experience.","job":"Junior ML Engineer. Python, PyTorch basics, willingness to learn, software engineering fundamentals."},

    # Easy negatives — cross-domain (kept but paired with varied jobs)
    {"id":"E11","label":0,"resume":"Frontend developer. React, TypeScript, CSS. Figma design systems. 0 ML experience.","job":"ML Platform Engineer. Embedding models, vector search, feature store, real-time serving, FAISS."},
    {"id":"E12","label":0,"resume":"Accountant. GAAP, Excel, financial modelling, tax compliance. CPA certified.","job":"MLOps Engineer. Kubernetes, model serving, monitoring, drift detection, CI/CD for ML."},
    {"id":"E13","label":0,"resume":"HR manager. Talent acquisition, performance reviews, HRIS systems, employee relations.","job":"AI Research Engineer. LLM fine-tuning, RLHF, distributed training, NLP, transformer architectures."},
    {"id":"E14","label":0,"resume":"Mobile developer. Swift, Kotlin, iOS/Android. App Store 4.8 rating, 200K downloads.","job":"AI Infra Engineer. LLM serving, vLLM, inference optimization, GPU utilization, high-throughput systems."},
    {"id":"E15","label":0,"resume":"Sales representative. B2B SaaS, quota 120%, Salesforce CRM, enterprise deal closing.","job":"Computer Vision Engineer. Object detection, segmentation, PyTorch, edge deployment, model optimization."},
    {"id":"E16","label":0,"resume":"DevOps engineer. Terraform, Ansible, Jenkins. Zero ML background.","job":"ML Research Scientist. Novel architecture design, paper publication, NeurIPS/ICML, deep learning research."},
    {"id":"E17","label":0,"resume":"Project manager. Agile, JIRA, stakeholder management, budget planning. No technical background.","job":"ML Research Scientist. Novel architecture design, paper publication, NeurIPS/ICML, PyTorch."},
    {"id":"E18","label":0,"resume":"QA engineer. Selenium, test automation, bug tracking. No ML exposure.","job":"Senior ML Engineer. 5+ years deep learning, PyTorch, production systems at scale."},
    {"id":"E19","label":0,"resume":"Marketing analyst. Google Analytics, A/B testing on web UI, Excel. No programming.","job":"Data Engineer. Kafka, Spark, BigQuery, real-time pipelines, feature engineering for ML."},
    {"id":"E20","label":0,"resume":"Mechanical engineer. CAD, FEA analysis, manufacturing tolerances. No software background.","job":"AI Infra Engineer. LLM serving, vLLM, inference optimization, GPU utilization."},

    # Hard negatives — same domain, wrong level/spec (these break AUC=1.0)
    {"id":"E21","label":0,"resume":"Data scientist, 1 year exp. Python, pandas, simple regression, Kaggle. No production ML.","job":"Senior ML Engineer. 7+ years PyTorch, distributed training, production systems, team leadership."},
    {"id":"E22","label":0,"resume":"DevOps engineer. Terraform, Ansible, Jenkins, CI/CD pipelines. No Python ML libs, no modeling.","job":"ML Engineer. PyTorch, distributed training, MLOps, recommendation systems, feature engineering."},
    {"id":"E23","label":0,"resume":"Quantum computing researcher. Physics simulations, C++, MATLAB, variational algorithms. Zero deployment experience.","job":"ML Platform Engineer. Embedding models, vector search, feature store, real-time serving, FAISS."},
    {"id":"E24","label":0,"resume":"BI analyst. SQL, Tableau, Power BI, Excel dashboards. Descriptive analytics only. No ML modeling.","job":"ML Research Scientist. Novel architecture design, deep learning research, NeurIPS/ICML publications."},
    {"id":"E25","label":0,"resume":"NLP researcher, academia only. BERT, sentiment analysis, text classification. Zero production or infra experience.","job":"MLOps Engineer. Kubernetes, model serving, monitoring, drift detection, CI/CD, 99.9% SLO."},
    {"id":"E31","label":0,"resume":"Cybersecurity engineer. Penetration testing, SIEM, zero-trust architecture. CISSP certified. Strong Python scripting.","job":"ML Engineer. PyTorch, distributed training, model deployment, recommendation systems."},
    {"id":"E32","label":0,"resume":"Game developer. Unity, C#, Unreal Engine, shader programming, GPU physics simulations. 5 shipped titles.","job":"AI Research Scientist. Novel deep learning architecture, NeurIPS publications, PyTorch, CUDA."},
    {"id":"E33","label":0,"resume":"Embedded systems engineer. C, RTOS, bare-metal, FPGA, IoT firmware. Strong low-level optimization.","job":"ML Platform Engineer. Embedding models, vector search, feature store, real-time serving."},
    {"id":"E34","label":0,"resume":"Network engineer. Cisco, BGP routing, SDN, Python network automation. CCIE certified. No ML background.","job":"AI Infra Engineer. LLM serving, vLLM, GPU optimization, high-throughput AI inference systems."},
    {"id":"E35","label":0,"resume":"Technical writer. API documentation, developer experience, Markdown, code examples. No ML or engineering background.","job":"Senior ML Engineer. 5+ years PyTorch, production deep learning systems, team leadership."},

    # Near-miss positives — weak matches that should rank lower than strong positives
    {"id":"E26","label":1,"resume":"ML engineer. PyTorch, basic model training and deployment. 2 years exp. Scikit-learn, some NLP work.","job":"ML Engineer. Python, model training and deployment, PyTorch, team collaboration."},
    {"id":"E27","label":1,"resume":"AI researcher. Transformer fine-tuning, Hugging Face, some RLHF experiments. 3 papers under review.","job":"AI Research Engineer. LLM fine-tuning, RLHF, NLP, research publications."},
    {"id":"E28","label":1,"resume":"Platform engineer. Built FAISS vector index, basic feature store, simple embedding models. Apache Spark experience.","job":"ML Platform Engineer. Embedding models, vector search, feature engineering, Spark."},
    {"id":"E29","label":1,"resume":"MLOps engineer. Docker, Kubernetes, basic model serving via FastAPI. Prometheus monitoring setup.","job":"MLOps Engineer. Kubernetes, model monitoring, CI/CD for ML, containerization."},
    {"id":"E30","label":1,"resume":"Data engineer. Kafka, Spark Streaming, PostgreSQL. Builds feature pipelines for ML teams. dbt experience.","job":"Data Engineer. Kafka, Spark, feature pipelines for ML, SQL, data quality monitoring."},

    # Strong positives — batch 2
    {"id":"E36","label":1,"resume":"Senior ML engineer. PyTorch, distributed training 128-GPU clusters, MLflow, FAISS. Led 4-engineer team. Production models for 10M daily users.","job":"ML Engineer — Recommendations. PyTorch, distributed training, MLOps, production at scale."},
    {"id":"E37","label":1,"resume":"LLM engineer. Fine-tuned LLaMA-2, RLHF with PPO, vLLM serving, speculative decoding. Reduced inference cost 35%.","job":"AI Research Engineer. LLM fine-tuning, RLHF, distributed training, transformer architectures."},
    {"id":"E38","label":1,"resume":"Applied scientist. XGBoost, deep learning, Spark feature engineering, A/B testing framework. 20+ experiments shipped to production.","job":"ML Engineer. Feature engineering, experiment tracking, model deployment, Python, Spark."},
    {"id":"E39","label":1,"resume":"Computer vision engineer. YOLO v8, SAM segmentation, edge deployment NVIDIA Jetson Orin. 96% mAP on production dataset.","job":"Computer Vision Engineer. Object detection, segmentation, PyTorch, edge deployment, optimization."},
    {"id":"E40","label":1,"resume":"AI infra engineer. Built vLLM serving cluster, KV cache optimization, 8x GPU utilization improvement. 50K QPS sustained throughput.","job":"AI Infra Engineer. LLM serving, vLLM, inference optimization, GPU utilization, high-throughput."},
]

def run_ir_eval(score_fn, system_name, K=5) -> dict:
    """Compute Precision@K, Recall@K, Top-K Acc, NDCG@K for a scoring function."""
    scored = sorted(
        [(score_fn(e["resume"], e["job"]), e["label"]) for e in EVAL_DATASET],
        key=lambda x: x[0],
        reverse=True
    )
    labels = [l for _, l in scored]
    n_rel = sum(labels)

    return {
        "System": system_name,
        f"P@{K}": round(precision_at_k(labels, K), 3),
        f"R@{K}": round(recall_at_k(labels, K, n_rel), 3),
        f"Top-{K} Acc": round(topk_accuracy(labels, K), 3),
        f"NDCG@{K}": round(ndcg_at_k(labels, K), 3),
    }

print(f"✅ Evaluator + IR dataset loaded ({len(EVAL_DATASET)} pairs)")


✅ Evaluator + IR dataset loaded (40 pairs)


---
## 🚀 CELL 11 — Build + Train All Models

In [13]:
import nest_asyncio; nest_asyncio.apply()
random.seed(42)

# ── 1. Job Corpus ───────────────────────────────────────────────────────────
CORPUS = JobCorpus(INGESTION)
JOB_CORPUS_DATA = [
    JobDocument("J01","ML Engineer — Recommendations","Meta","PyTorch, distributed training, MLOps, A/B testing, ranking, two-tower architecture, FAISS.","senior","ml"),
    JobDocument("J02","AI Research Engineer","DeepMind","LLM fine-tuning, RLHF, distributed training, NLP, transformer architectures, Hugging Face.","senior","research"),
    JobDocument("J03","MLOps Engineer","Uber","Model serving, Kubernetes, monitoring, drift detection, CI/CD for ML, Kubeflow, MLflow.","mid","infra"),
    JobDocument("J04","ML Platform Engineer","Airbnb","Embedding models, vector search, feature store, real-time serving, FAISS, Apache Flink.","senior","ml"),
    JobDocument("J05","AI Infra Engineer","Google","LLM serving, vLLM, inference optimization, GPU utilization, speculative decoding, TensorRT.","senior","infra"),
    JobDocument("J06","Data Engineer","Netflix","Kafka, Spark, BigQuery, real-time pipelines, feature engineering, dbt, Airflow.","mid","data"),
    JobDocument("J07","Computer Vision Engineer","Tesla","Object detection, segmentation, PyTorch, edge deployment, NVIDIA Jetson, model optimization.","mid","ml"),
    JobDocument("J08","Senior ML Engineer","Spotify","Recommendation systems, collaborative filtering, feature engineering, Python, PyTorch, A/B testing.","senior","ml"),
    JobDocument("J09","Junior ML Engineer","Startup","Python, PyTorch basics, willingness to learn, software engineering fundamentals, scikit-learn.","junior","ml"),
    JobDocument("J10","ML Research Scientist","OpenAI","Novel architecture design, paper publication, NeurIPS/ICML, deep learning research, PyTorch.","senior","research"),
]
CORPUS.add_batch(JOB_CORPUS_DATA)

# ── 2. Train Contrastive Ranker ─────────────────────────────────────────────
# Separate train/eval IDs — hard negatives (E21-E35) reserved for eval only
TRAIN_IDS = {"E01","E02","E03","E04","E05","E06","E07","E08","E09","E10",
             "E11","E12","E13","E14","E15","E16","E17","E18","E19","E20",
             "E36","E37","E38","E39","E40"}
EVAL_IDS  = {"E21","E22","E23","E24","E25","E26","E27","E28","E29","E30",
             "E31","E32","E33","E34","E35"}

train_pool = [e for e in EVAL_DATASET if e["id"] in TRAIN_IDS]
eval_pool  = [e for e in EVAL_DATASET if e["id"] in EVAL_IDS]

positives = [e for e in train_pool if e["label"] == 1]
negatives  = [e for e in train_pool if e["label"] == 0]
train_r, train_pos, train_neg = [], [], []
for pos in positives:
    neg = random.choice(negatives)
    train_r.append(INGESTION.embed(pos["resume"]))
    train_pos.append(INGESTION.embed(pos["job"]))
    train_neg.append(INGESTION.embed(neg["job"]))

# Hard negatives for training — same-domain wrong level
HARD_NEG_PAIRS = [
    (positives[0]["resume"], positives[0]["job"],
     "ML Research Scientist. Novel architecture design, NeurIPS/ICML publications, theoretical ML."),
    (positives[1]["resume"], positives[1]["job"],
     "Senior ML Engineer. 10+ years, team leadership, staff-level system design, cross-org."),
    (positives[2]["resume"], positives[2]["job"],
     "Data Science Manager. People management, roadmap planning, stakeholder alignment."),
    (positives[3]["resume"], positives[3]["job"],
     "AI Safety Researcher. Alignment research, interpretability, formal verification."),
    (positives[4]["resume"], positives[4]["job"],
     "Quantum ML Researcher. Quantum circuits, variational algorithms, qiskit."),
    (positives[0]["resume"], positives[0]["job"],
     "Frontend Engineer. React, TypeScript, CSS, UI/UX, design systems."),
    (positives[1]["resume"], positives[1]["job"],
     "Accountant. GAAP, financial modelling, tax compliance, audit."),
    (positives[2]["resume"], positives[2]["job"],
     "HR Manager. Talent acquisition, HRIS, employee relations, compliance."),
]
for pos_r, pos_j, neg_j in HARD_NEG_PAIRS:
    train_r.append(INGESTION.embed(pos_r))
    train_pos.append(INGESTION.embed(pos_j))
    train_neg.append(INGESTION.embed(neg_j))
console.print(f"[dim]Training set: {len(train_r)} triplets | Eval set: {len(eval_pool)} pairs (all hard)[/dim]")

TRAINER = ContrastiveTrainer(emb_dim=EMBED_DIM, margin=0.3, lr=1e-3, epochs=RANKER_EPOCHS)
TRAINER.fit(PairwiseDataset(np.vstack(train_r), np.vstack(train_pos), np.vstack(train_neg)), batch_size=8)

# ── 3. Calibrate Ranker — fit on train_pool, eval on hard eval_pool only ────
console.rule("[bold cyan]Ranker Calibration + Classification Metrics[/bold cyan]")
random.seed(42)
_calib_train = train_pool                      # easy + strong positives for fitting Platt scaler
_calib_eval  = eval_pool                       # ONLY hard negatives + near-miss positives for reporting
RANKER_METRICS = TRAINER.calibrate(_calib_eval, INGESTION, fit_on=_calib_train)
console.print(f"[dim]Calibration: {len(_calib_train)} fit / {len(_calib_eval)} eval (hard negatives only)[/dim]")

# ── 4. Train Learned Hallucination Detector ─────────────────────────────────
console.rule("[bold red]Training Learned Hallucination Detector[/bold red]")
DETECTOR = LearnedHallucinationDetector(INGESTION)
# Use ALL positive resumes as source facts (not just first 10) for richer synthetic pairs
source_facts = [e["resume"] for e in EVAL_DATASET if e["label"] == 1]
HALLUC_REPORT = DETECTOR.train(source_facts)

# ── 5. Build remaining modules ──────────────────────────────────────────────
SCORER    = EnsembleScorer(INGESTION, CORPUS, TRAINER)
VALIDATOR = ConfidenceGatedValidator(DETECTOR, INGESTION)
PARSER    = ResumeParser()
OPTIMIZER = ControlledOptimizer()

console.print(Panel(
    "[bold green]✅ System Ready[/bold green]\n"
    "ResumeParser → ContrastiveRanker (calibrated, AUC/F1) → ControlledOptimizer\n"
    "→ LearnedHallucinationDetector → ConfidenceGatedValidator → GradientExplainer",
    title="🏆 Job Hunt Agent"
))
print(INGESTION.cache_stats())

✅ Indexed 10 jobs into FAISS

Training set: 23 triplets | Eval set: 15 pairs (all hard)

Training contrastive ranker — 23 pairs, 12 epochs

Epoch 03/12 — loss 0.0459

Epoch 06/12 — loss 0.0297

...

  Hallucination Detector — Synthetic Test Set Metrics  
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━┓
┃ Class        ┃ Precision ┃ Recall ┃ F1    ┃ Support ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━┩
│ grounded     │ 0.947     │ 0.900  │ 0.923 │ 20      │
│ hallucinated │ 0.875     │ 0.933  │ 0.903 │ 15      │
└──────────────┴───────────┴────────┴───────┴─────────┘

  Hallucination Detector — Curated Val Set   
              (n=15, AUC=0.980)              
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃ Class        ┃ Precision ┃ Recall ┃ F1    ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ grounded     │ 1.000     │ 0.700  │ 0.824 │
│ hallucinated │ 0.625     │ 1.000  │ 0.769 │
└──────────────┴───────────┴────────┴───────┘

Curated set overall accuracy: 0.800 | AUC: 0.980

╭─────────────────────────────────────────────── 🏆 Job Hunt Agent ───────────────────────────────────────────────╮
│ ✅ System Ready                                                                                                 │
│ ResumeParser → ContrastiveRanker (calibrated, AUC/F1) → ControlledOptimizer                                     │
│ → LearnedHallucinationDetector → ConfidenceGatedValidator → GradientExplainer                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Cache: 243 hits / 226 misses (51.8% hit rate)


In [15]:
RESUME_PDF_PATH = "path/to/your_resume.pdf"   # ← SET THIS

JOB_DESCRIPTION = """
AI Engineer I | American Express — Agentic AI

You will be a hands-on builder contributing to production agentic AI systems.

Responsibilities:
- Contribute to LLM-powered and agentic product features.
- Build agentic AI workflows: context reasoning, tool calling, actions.
- Implement and maintain RAG pipelines over financial data.
- Contribute to shared AI infrastructure: LLM services, orchestration, evaluation.
- Operate AI systems in production: monitoring, debugging, reliability.

Technical Stack: Python, Go, TypeScript | AWS/GCP, Kubernetes | REST, gRPC
Agentic AI: LLMs in agentic workflows, RAG, vector storage, agent orchestration, evaluation
Schema validation, structured data handling

Qualifications:
- 1+ years professional software engineering experience
- Hands-on experience with LLM-based applications or applied ML systems
- Solid backend engineering fundamentals (Python preferred)
- Interest in agentic AI systems and AI-assisted development workflows

Preferred: LLM tooling, prompt engineering, RAG, agent frameworks, open-source contributions
"""

# ── Demo inline resume if PDF missing ───────────────────────────────────────
RESUME_TEXT_OVERRIDE = """
Bassam Nazer | Chennai, Tamil Nadu
AI/ML Engineer with end-to-end experience in deep learning, medical imaging, and LLM-powered RAG systems.
Built a production chest X-ray classifier (14 pathologies, 0.92 recall) in clinical collaboration with ICMR-NIRT radiologists.
Published at IEEE CVMI 2025. Recognized among 196 global TB innovations by the Gates Foundation.

EXPERIENCE:
C-DAC Chennai | Project Associate – AI/ML Engineer | Aug 2024 – Present
- Designed DenseNet201 pipeline for multi-label chest X-ray classification, 14 pathologies, 0.92 recall, 0.86 specificity
- Implemented anatomical zone segmentation (CXAS model) across six lung regions with GradCAM saliency maps
- Addressed class imbalance using focal loss and weighted binary cross-entropy
- Deployed Flask REST APIs on NVIDIA Quadro multi-GPU systems with result caching
- Field-deployed in National Health Mission Chennai mobile TB screening ambulances

PROJECTS:
Hierarchical RAG System | Python, FAISS, LangChain, HuggingFace | 2026
- Two-stage hierarchical retrieval over 100+ PDFs / 10,000+ indexed chunks
- Reduced irrelevant context by 40% vs flat retrieval baseline
- Sub-100ms retrieval latency on single CPU node
- BAAI/bge-large-en embeddings; constrained generation (temperature=0)

Clinical LLM Agent with Tool-Augmented RAG | Python, PubMed API, FAISS, LangChain | 2026
- LangGraph-based agentic pipeline: PubMed API search, FAISS indexing, semantic retrieval
- nomic-embed-text embeddings; citation-grounded prompting; fault-tolerant tool dispatcher

PUBLICATIONS:
- IEEE CVMI 2025: Data Poisoning Attacks and Defenses in Machine Learning
- CML 2026: Q3D-MRI-Net: Hybrid Quantum-Classical 3D Network for Alzheimer's Detection

AWARDS:
- XrayCAD selected among 196 global TB innovations (Gates Foundation / UNION)
- Award for Research in AI-Assisted TB Detection, ICMR-NIRT Chennai 2025

EDUCATION: M.Tech, Data Science, DIAT Pune, 2024, CGPA 8.16

SKILLS: PyTorch, TensorFlow, Keras, Scikit-learn, Hugging Face Transformers, FAISS, LangChain, LangGraph,
BAAI/bge-large-en, nomic-embed-text, DenseNet201, GradCAM, Flask, Docker, NVIDIA Quadro,
OpenCV, Python, SQL, NumPy, Pandas, Matplotlib, PostgreSQL, MySQL, Git, Linux,
focal loss, RAG, agentic pipelines, prompt engineering, JSON schema,
hallucination detection, constrained generation, reranking, citation-grounded generation
"""

pdf_path = pathlib.Path(RESUME_PDF_PATH)
if pdf_path.exists():
    RESUME_TEXT = INGESTION.parse_pdf(RESUME_PDF_PATH)
    console.print(f"[green]✅ PDF loaded: {len(RESUME_TEXT.split())} words[/green]")
else:
    RESUME_TEXT = RESUME_TEXT_OVERRIDE
    console.print("[yellow]⚠ Using inline resume (set RESUME_PDF_PATH to use your PDF)[/yellow]")

print(f"✅ JD: {len(JOB_DESCRIPTION.split())} words")


✅ PDF loaded: 906 words

✅ JD: 141 words


---
## 📄 CELL 12 — Configure Inputs

In [16]:
# ── Stage 1: Structured Fact Extraction ─────────────────────────────────────
console.rule("[bold cyan]Stage 1: Structured Fact Extraction[/bold cyan]")
FACTSHEET = PARSER.extract(RESUME_TEXT)  
console.print(Panel(PARSER.summary(FACTSHEET), title="📋 FactSheet"))
console.print(f"[dim]Skills ({len(FACTSHEET.skills)}): {', '.join(FACTSHEET.skills[:12])}...[/dim]")

# ── Ensemble Scoring + Explainability ───────────────────────────────────────
console.rule("[bold cyan]Ensemble Scoring + Gradient Attribution[/bold cyan]")
score_result = SCORER.score(RESUME_TEXT, JOB_DESCRIPTION)
console.print(Panel(
    f"Composite  : [bold]{score_result['composite']}%[/bold]\n"
    f"  Cosine (30%)          : {score_result['cosine']}%\n"
    f"  Calibrated Ranker (55%): {score_result['ranker']}%\n"
    f"  RAG Boost (15%)       : {'✅ in top-5' if score_result['rag_boost'] else '❌ not retrieved'}\n"
    f"Uncertainty: {score_result.get('uncertainty','N/A')}",
    title="📊 Match Score", border_style="yellow"
))
if "resume_drivers" in score_result:
    console.print("[bold]Why this resume matches (top gradient drivers):[/bold]")
    for s in score_result["resume_drivers"]:
        console.print(f"  → {s[:100]}")

# ── Stage 2: Controlled Optimization ────────────────────────────────────────
console.rule("[bold cyan]Stage 2: Controlled Optimization (FactSheet Only)[/bold cyan]")
OPT_RESUME = OPTIMIZER.optimize(FACTSHEET, JOB_DESCRIPTION)
console.print(Panel(OPT_RESUME.summary, title="📝 Generated Summary"))
console.print("\n[bold]Evidenced Bullets (before validation):[/bold]")
for i, eb in enumerate(OPT_RESUME.evidenced_bullets, 1):
    console.print(f"  [bold cyan]{i}.[/bold cyan] {eb.bullet}")
    console.print(f"     [dim]↳ Source: {eb.source_fact[:90]}[/dim]")

print("\n✅ Two-stage pipeline complete")


─────────────────────────────────────── Stage 1: Structured Fact Extraction ───────────────────────────────────────

╭───────────────────────────────────────────────── 📋 FactSheet ──────────────────────────────────────────────────╮
│ Candidate: Bassam Nazer | Skills: 43 | Experiences: 1 | Projects: 3 | Metrics: 25 | Publications: 2             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Skills (43): Python, SQL, PyTorch, TensorFlow, Keras, Scikit-learn, CNNs, Transformers, Multi-label Classification,
Segmentation, Explainable AI (GradCAM), Focal Loss...

───────────────────────────────────── Ensemble Scoring + Gradient Attribution ─────────────────────────────────────

╭──────────────────────────────────────────────── 📊 Match Score ─────────────────────────────────────────────────╮
│ Composite  : 38.4%                                                                                              │
│   Cosine (30%)          : 43.4%                                                                                 │
│   Calibrated Ranker (55%): 46.1%                                                                                │
│   RAG Boost (15%)       : ❌ not retrieved                                                                      │
│ Uncertainty: mean=0.310 ± 0.079  95% CI [0.214, 0.487]                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

7. Engineered a PyTorch pairwise contrastive ranker over a 40-pair IR benchmark, where the ensemble scorer 
achieved NDCG@5 = 1.0 and Precision@5 = 1.0.

↳ Source: Engineered a PyTorch pairwise contrastive ranker (MarginRankingLoss, MC Dropout uncertaint

8. Constrained LLM generation to retrieved context only (temperature=0) with a fault-tolerant tool dispatcher and
JSON fallback to handle malformed LLM outputs.

↳ Source: Designed a FAISS retrieval layer with nomic-embed-text embeddings over live PubMed abstrac


✅ Two-stage pipeline complete


---
## 🚀 CELL 13 — Run the Full Pipeline

In [17]:
# ── REGENERATION LOOP ──────────────────────────────────────────────
# Run initial validation, then retry rejected bullets up to MAX_REGEN_ATTEMPTS times
# Each retry uses a progressively stricter prompt (see REGEN_PROMPTS in Cell 16)

console.rule("[bold red] Regeneration Loop[/bold red]")

# Initial validation pass
reports_init, OPT_RESUME = VALIDATOR.validate_and_gate(OPT_RESUME)
n_init_rejected = sum(1 for r in reports_init if r.final_verdict)
console.print(f"Initial validation: {n_init_rejected}/{len(reports_init)} bullets rejected → entering regeneration loop")

for attempt in range(MAX_REGEN_ATTEMPTS):
    still_rejected = [eb for eb in OPT_RESUME.evidenced_bullets if eb.rejected]
    if not still_rejected:
        console.print(f"[bold green]✅ All bullets pass after {attempt} regeneration attempt(s)[/bold green]")
        break

    console.print(f"\n[yellow]Attempt {attempt+1}/{MAX_REGEN_ATTEMPTS} — regenerating {len(still_rejected)} bullet(s)...[/yellow]")

    for eb in still_rejected:
        new_eb = OPTIMIZER.regenerate_bullet(
            rejected_bullet=eb.bullet,
            source_fact=eb.source_fact,
            job_description=JOB_DESCRIPTION,
            attempt=attempt,
        )
        # Re-validate the regenerated bullet
        report_new = VALIDATOR.validate_bullet(new_eb, OPT_RESUME.factsheet_snapshot)
        new_eb.p_hallucination = report_new.p_hallucination
        new_eb.rejected = report_new.final_verdict

        # Replace in-place
        idx = OPT_RESUME.evidenced_bullets.index(eb)
        OPT_RESUME.evidenced_bullets[idx] = new_eb

        status = "[red]STILL REJECTED[/red]" if new_eb.rejected else "[green]NOW PASSED[/green]"
        console.print(f"  → {new_eb.bullet[:70]}… [{status}]")

# Final attempt: discard any bullets that still fail after all retries
still_failing = [eb for eb in OPT_RESUME.evidenced_bullets if eb.rejected]
if still_failing:
    console.print(f"\n[bold red]⚠ {len(still_failing)} bullet(s) discarded after {MAX_REGEN_ATTEMPTS} retries[/bold red]")
    for eb in still_failing:
        console.print(f"  [red]Discarded:[/red] {eb.bullet[:80]}…")

# Final validation report
reports_final, OPT_RESUME = VALIDATOR.validate_and_gate(OPT_RESUME)
n_final_rejected = sum(1 for r in reports_final if r.final_verdict)
console.print(f"\n[bold]Regeneration loop complete:[/bold] {n_init_rejected} initial rejections → {n_final_rejected} final rejections")
console.print(f"Hallucination rate: {VALIDATOR.hallucination_rate(reports_final):.1%} (target < 5%)")
reports = reports_final  # use final reports downstream


───────────────────────────────────────────────  Regeneration Loop ────────────────────────────────────────────────

Initial validation: 2/8 bullets rejected → entering regeneration loop

Attempt 1/3 — regenerating 2 bullet(s)...

→ Deployed an inference pipeline via Flask REST APIs on NVIDIA Quadro mu… [STILL REJECTED]

→ Designed a FAISS retrieval layer with nomic-embed-text embeddings over… [NOW PASSED]

Attempt 2/3 — regenerating 1 bullet(s)...

→ Addressed severe class imbalance using focal loss and weighted binary … [NOW PASSED]

✅ All bullets pass after 2 regeneration attempt(s)

Regeneration loop complete: 2 initial rejections → 0 final rejections

Hallucination rate: 0.0% (target < 5%)

---
## 🔍 CELL 14 — Two-Layer Hallucination Validation + Confidence Gate

In [18]:
console.rule("[bold red]Two-Layer Hallucination Validation[/bold red]")

reports, OPT_RESUME = VALIDATOR.validate_and_gate(OPT_RESUME)
halluc_rate = VALIDATOR.hallucination_rate(reports)
avg_faith   = VALIDATOR.avg_faithfulness(reports)

table = Table(title="Per-Bullet Hallucination Report", header_style="bold")
table.add_column("#", width=3)
table.add_column("Bullet", width=45)
table.add_column("P(halluc)", width=9)
table.add_column("Learned", width=8)
table.add_column("Rules", width=8)
table.add_column("Faithfulness", width=12)
table.add_column("Verdict", width=8)

for i, r in enumerate(reports, 1):
    verdict = "[red]❌ REJECT[/red]" if r.final_verdict else "[green]✅ PASS[/green]"
    table.add_row(
        str(i),
        r.bullet[:45] + "…" if len(r.bullet)>45 else r.bullet,
        f"{r.p_hallucination:.3f}",
        "[red]YES[/red]" if r.is_learned_halluc else "[green]NO[/green]",
        r.rule_halluc_type if r.rule_flags else "—",
        f"{r.faithfulness_cosine:.3f}",
        verdict,
    )
console.print(table)

n_passed   = sum(1 for r in reports if not r.final_verdict)
n_rejected = sum(1 for r in reports if r.final_verdict)
console.print(f"\n[bold]Summary:[/bold] {n_passed} passed · {n_rejected} rejected · "
              f"Hallucination rate: {halluc_rate:.1%} · Avg faithfulness: {avg_faith:.3f}")

# Show accepted bullets only
if n_rejected > 0:
    console.print("\n[yellow]Accepted bullets (after confidence gating):[/yellow]")
    for i, eb in enumerate(OPT_RESUME.evidenced_bullets, 1):
        if not eb.rejected:
            console.print(f"  {i}. {eb.bullet}")


─────────────────────────────────────── Two-Layer Hallucination Validation ────────────────────────────────────────

                                          Per-Bullet Hallucination Report                                          
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ #   ┃ Bullet                                        ┃ P(halluc) ┃ Learned  ┃ Rules    ┃ Faithfulness ┃ Verdict  ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ 1   │ Built a LangGraph-based agentic pipeline      │ 0.429     │ NO       │ skill    │ 0.940        │ ✅ PASS  │
│     │ orch…                                         │           │          │          │              │          │
│ 2   │ Developed a two-stage hierarchical retrieval  │ 0.471     │ NO       │ skill    │ 0.926        │ ✅ PASS  │
│     │ …                                             │           │          │          │              │          │
│ 3   │ Reduced hallucination rate from ~28% to 0%    │ 0.448     │ NO       │ skill    │ 0.880        │ ✅ PASS  │
│     │ by…                                           │           │          │          │              │          │
│ 4   │ Addressed severe class imbalance using focal  │ 0.413     │ NO       │ skill    │ 0.992        │ ✅ PASS  │
│     │ …                                             │           │          │          │              │          │
│ 5   │ Designed and trained a DenseNet201 deep       │ 0.437     │ NO       │ skill    │ 0.917        │ ✅ PASS  │
│     │ learn…                                        │           │          │          │              │          │
│ 6   │ Trained a learned binary hallucination        │ 0.444     │ NO       │ skill    │ 0.891        │ ✅ PASS  │
│     │ detect…                                       │           │          │          │              │          │
│ 7   │ Engineered a PyTorch pairwise contrastive     │ 0.442     │ NO       │ skill    │ 0.897        │ ✅ PASS  │
│     │ ran…                                          │           │          │          │              │          │
│ 8   │ Designed a FAISS retrieval layer with         │ 0.430     │ NO       │ skill    │ 0.937        │ ✅ PASS  │
│     │ nomic-e…                                      │           │          │          │              │          │
└─────┴───────────────────────────────────────────────┴───────────┴──────────┴──────────┴──────────────┴──────────┘

Summary: 8 passed · 0 rejected · Hallucination rate: 0.0% · Avg faithfulness: 0.923

In [19]:
import scipy.stats
console.rule("[bold magenta]Full Evaluation Suite[/bold magenta]")

# ── Grounding metrics ─────────────────────────────────────────
opt_text = OPT_RESUME.summary + " " + " ".join(
    eb.bullet for eb in OPT_RESUME.evidenced_bullets if not eb.rejected
)
orig_emb = INGESTION.embed(RESUME_TEXT)
opt_emb  = INGESTION.embed(opt_text)
jd_emb   = INGESTION.embed(JOB_DESCRIPTION)

improved = EvalResult(
    hallucination_rate  = halluc_rate,
    faithfulness_score  = avg_faith,
    semantic_similarity = float(np.dot(orig_emb, opt_emb)),
    jd_relevance        = float(np.dot(opt_emb, jd_emb)),
    total_bullets       = len(reports),
    flagged_bullets     = n_rejected,
)
improved.display("Grounding Evaluation")

# ── Baseline (unconstrained — no FactSheet) ──────────────────────────────────
# Measured: run the optimizer WITHOUT FactSheet constraint (raw text prompt)
BASELINE_PROMPT = """Tailor this resume for the job description.
Rewrite the resume bullets to match the JD. You may add relevant skills and rephrase freely.
Resume: {resume}
JD: {jd}
Return JSON: {{"summary": "...", "bullets": [{{"bullet":"...", "source_fact":"..."}}]}}"""

from google import genai
_gclient = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
_bl_resp  = _gclient.models.generate_content(
    model=LLM_MODEL_NAME,
    contents=BASELINE_PROMPT.format(resume=RESUME_TEXT[:1500], jd=JOB_DESCRIPTION[:1500]),
    config={"temperature": 0.7},   # unconstrained — more creative/hallucination-prone
)
try:
    _bl_raw     = re.sub(r'^```json\s*','',_bl_resp.text.strip())
    _bl_raw     = re.sub(r'```\s*$','',_bl_raw)
    _bl_data    = json.loads(_bl_raw)
    _bl_bullets = [EvidencedBullet(b["bullet"], b.get("source_fact","")) for b in _bl_data.get("bullets",[])]
except Exception:
    _bl_bullets = [EvidencedBullet("Fallback baseline bullet.", "")]

# Measure baseline hallucination rate with the learned detector
_bl_snap = {"skills": FACTSHEET.skills, "metrics": [], "organizations": []}
_bl_reports = [VALIDATOR.validate_bullet(eb, _bl_snap) for eb in _bl_bullets]
_bl_halluc  = VALIDATOR.hallucination_rate(_bl_reports) if _bl_reports else 0.28
_bl_faith   = VALIDATOR.avg_faithfulness(_bl_reports)   if _bl_reports else 0.47

_bl_text  = " ".join(eb.bullet for eb in _bl_bullets)
_bl_emb   = INGESTION.embed(_bl_text) if _bl_text.strip() else orig_emb
_bl_semsim = float(np.dot(orig_emb, _bl_emb))
_bl_jdrel  = float(np.dot(_bl_emb,  jd_emb))

baseline = EvalResult(
    hallucination_rate  = _bl_halluc,
    faithfulness_score  = _bl_faith,
    semantic_similarity = _bl_semsim,
    jd_relevance        = _bl_jdrel,
    total_bullets       = len(_bl_bullets),
    flagged_bullets     = sum(1 for r in _bl_reports if r.final_verdict),
)

# ── Comparison Table ─────────────────────────────────────────────────────────
cmp = Table(title="📊 Baseline vs Previous Version — All Metrics", header_style="bold cyan")
cmp.add_column("Metric"); cmp.add_column("Baseline (unconstrained)"); cmp.add_column("Previous Version (FactSheet+Detector)"); cmp.add_column("Δ")

def delta_str(b, i, lower_better=False):
    d = i - b
    better = (d < 0) if lower_better else (d > 0)
    sign = "+" if d >= 0 else ""
    color = "green" if better else "red"
    return f"[{color}]{sign}{d:.3f}[/{color}]"

rows = [
    ("Hallucination Rate",  f"{baseline.hallucination_rate:.1%}", f"{improved.hallucination_rate:.1%}",
     delta_str(baseline.hallucination_rate, improved.hallucination_rate, lower_better=True)),
    ("Faithfulness Score",  f"{baseline.faithfulness_score:.3f}", f"{improved.faithfulness_score:.3f}",
     delta_str(baseline.faithfulness_score, improved.faithfulness_score)),
    ("Semantic Similarity", f"{baseline.semantic_similarity:.3f}", f"{improved.semantic_similarity:.3f}",
     delta_str(baseline.semantic_similarity, improved.semantic_similarity)),
    ("JD Relevance",        f"{baseline.jd_relevance:.3f}", f"{improved.jd_relevance:.3f}",
     delta_str(baseline.jd_relevance, improved.jd_relevance)),
    ("Flagged Bullets",     f"{baseline.flagged_bullets}/{baseline.total_bullets}",
                             f"{improved.flagged_bullets}/{improved.total_bullets}", ""),
]
for row in rows: cmp.add_row(*row)
console.print(cmp)

# ── Ranker Classification Metrics ────────────────────────────────────────────
console.print("\n[bold]Ranker Classification Metrics (Platt-calibrated, threshold=0.5):[/bold]")
for k, v in RANKER_METRICS.items():
    console.print(f"  {k}: {v}")


────────────────────────────────────────────── Full Evaluation Suite ──────────────────────────────────────────────

          Grounding Evaluation          
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━┓
┃ Metric              ┃ Value ┃ Target ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━┩
│ Hallucination Rate  │ 0.0%  │ < 5%   │
│ Faithfulness Score  │ 0.923 │ > 0.65 │
│ Semantic Similarity │ 0.881 │ > 0.70 │
│ JD Relevance        │ 0.417 │ > 0.55 │
│ Flagged Bullets     │ 0/8   │ 0      │
└─────────────────────┴───────┴────────┘

                           📊 Baseline vs Previous Version — All Metrics                           
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Metric              ┃ Baseline (unconstrained) ┃ Previous Version (FactSheet+Detector) ┃ Δ      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Hallucination Rate  │ 100.0%                   │ 0.0%                                  │ -1.000 │
│ Faithfulness Score  │ 0.509                    │ 0.923                                 │ +0.413 │
│ Semantic Similarity │ 0.670                    │ 0.881                                 │ +0.211 │
│ JD Relevance        │ 0.631                    │ 0.417                                 │ -0.214 │
│ Flagged Bullets     │ 4/4                      │ 0/8                                   │        │
└─────────────────────┴──────────────────────────┴───────────────────────────────────────┴────────┘

Ranker Classification Metrics (Platt-calibrated, threshold=0.5):

AUC-ROC: 0.68

Accuracy: 0.667

Precision: 0.5

Recall: 0.6

F1: 0.545

---
## 📊 CELL 16 — IR Retrieval Evaluation (Per-Query, Redesigned)


In [20]:
# ── MODULE J (REVISED) — PER-QUERY IR EVALUATION ─────────────────────────────────────────────
#
# Problem with the original run_ir_eval():
#   All 40 pairs are ranked in a single global list. The 15 strong positives
#   (E01-E10, E36-E40) are so far from accountants and HR managers that ANY
#   embedding will push them to the top 5. All three systems scored P@5=1.0,
#   NDCG@5=1.0 -- indistinguishable from each other.
#
# Fix: Per-query retrieval evaluation.
#   For each of 8 job roles, a dedicated candidate pool is constructed:
#     * 1-2 strong positives      (relevance = 2)
#     * 0-2 near-miss/weak match  (relevance = 1)  <- graded, not binary
#     * 2   hard negatives        (relevance = 0)  <- same-domain confusables
#   The system must rank WITHIN this pool. No easy cross-domain negatives.
#
# Metrics (macro-averaged over 8 queries):
#   NDCG@K  -- graded relevance; rewards placing rel=2 above rel=1 above rel=0
#   MRR     -- mean reciprocal rank of first relevant result (rel > 0)
#   MAP@K   -- mean average precision; rewards both precision and recall @K
#
# Interpretation:
#   NDCG@K == 1.0 -> perfect ordering on every query
#   MRR == 1.0    -> always ranks a relevant resume #1
#   Any system that cannot distinguish E25 (NLP academic) from E04 (MLOps)
#   in Q03 will take a real NDCG hit -- unlike the old eval where it was masked.

import math

# ── Graded query pools ────────────────────────────────────────────────
# Each query is a job role with a curated candidate pool.
# Distractors are same-domain confusables, NOT easy cross-domain negatives.
IR_QUERIES = [
    {
        "qid": "Q01", "label": "ML Recommendations",
        "job": "ML Engineer - Recommendations. PyTorch, distributed training, MLOps, A/B testing, two-tower architecture, ranking systems.",
        "candidates": [
            ("E01", 2),   # Strong: PyTorch, FAISS, two-tower, A/B, 32-GPU
            ("E36", 2),   # Strong: PyTorch, FAISS, 128-GPU, 10M users
            ("E09", 1),   # Near-miss: Python/Docker/Kubernetes, some ML integration
            ("E26", 1),   # Near-miss: PyTorch basics, 2yr, scikit-learn
            ("E21", 0),   # Hard neg: junior DS, pandas/Kaggle, no production
            ("E22", 0),   # Hard neg: DevOps -- CI/CD overlaps MLOps vocabulary
        ],
    },
    {
        "qid": "Q02", "label": "AI Research Eng",
        "job": "AI Research Engineer. LLM fine-tuning, RLHF, distributed training, NLP, transformer architectures.",
        "candidates": [
            ("E02", 2),   # Strong: BERT fine-tuning, RLHF, 64-GPU, EMNLP
            ("E37", 2),   # Strong: LLaMA-2, RLHF+PPO, vLLM, speculative decoding
            ("E27", 1),   # Near-miss: Transformer fine-tuning, HuggingFace, some RLHF
            ("E24", 0),   # Hard neg: BI analyst -- SQL/Tableau, no ML modeling
            ("E32", 0),   # Hard neg: Game dev -- GPU experience but no DL/research
        ],
    },
    {
        "qid": "Q03", "label": "MLOps Engineer",
        "job": "MLOps Engineer. Model serving, Kubernetes, monitoring, drift detection, CI/CD for ML.",
        "candidates": [
            ("E04", 2),   # Strong: Kubernetes, Kubeflow, drift detection, 40 prod models
            ("E29", 2),   # Strong: Docker, Kubernetes, FastAPI serving, Prometheus
            ("E22", 0),   # Hard neg: DevOps -- Terraform/Jenkins but zero ML modeling
            ("E25", 0),   # Hard neg: NLP researcher -- has BERT/ML vocab, zero infra
        ],
    },
    {
        "qid": "Q04", "label": "ML Platform Eng",
        "job": "ML Platform Engineer. Embedding models, vector search, feature store, real-time serving.",
        "candidates": [
            ("E05", 2),   # Strong: FAISS, two-tower, real-time <5ms, 50+ A/B
            ("E28", 2),   # Strong: FAISS index, feature store, embedding models, Spark
            ("E23", 0),   # Hard neg: Quantum researcher -- C++/physics, no deployment
            ("E33", 0),   # Hard neg: Embedded sys -- FPGA/RTOS, similar low-level framing
        ],
    },
    {
        "qid": "Q05", "label": "AI Infra Engineer",
        "job": "AI Infra Engineer. LLM serving, vLLM, inference optimization, GPU utilization, high-throughput systems.",
        "candidates": [
            ("E07", 2),   # Strong: vLLM, speculative decoding, KV cache, 10K QPS
            ("E40", 2),   # Strong: vLLM cluster, KV cache, 8x GPU util, 50K QPS
            ("E34", 0),   # Hard neg: Network eng -- Python scripting, SDN infra
            ("E31", 0),   # Hard neg: Cybersec -- Python scripting, infra mindset
        ],
    },
    {
        "qid": "Q06", "label": "Computer Vision",
        "job": "Computer Vision Engineer. Object detection, segmentation, PyTorch, edge deployment, model optimization.",
        "candidates": [
            ("E06", 2),   # Strong: YOLO, segmentation, Jetson, 94% mAP
            ("E39", 2),   # Strong: YOLO v8, SAM, Jetson Orin, 96% mAP
            ("E35", 0),   # Hard neg: Tech writer -- API docs, no engineering
            ("E32", 0),   # Hard neg: Game dev -- GPU/shader, superficially similar
        ],
    },
    {
        "qid": "Q07", "label": "Data Engineer",
        "job": "Data Engineer. Kafka, Spark, BigQuery, real-time pipelines, feature engineering for ML.",
        "candidates": [
            ("E08", 2),   # Strong: Kafka, Spark, dbt, BigQuery, 1M events/sec
            ("E30", 2),   # Strong: Kafka, Spark Streaming, dbt, ML feature pipelines
            ("E24", 0),   # Hard neg: BI analyst -- SQL/Tableau, descriptive only
            ("E19", 0),   # Hard neg: Marketing analyst -- Excel/GA, no programming
        ],
    },
    {
        "qid": "Q08", "label": "Junior ML Engineer",
        "job": "Junior ML Engineer. Python, PyTorch basics, willingness to learn, software engineering fundamentals.",
        "candidates": [
            ("E10", 2),   # Strong: PyTorch tutorials, scikit-learn, 2yr SWE
            ("E26", 1),   # Near-miss: PyTorch deployed, 2yr -- slightly overqualified
            ("E21", 0),   # Hard neg: junior DS -- regression/Kaggle, no SWE depth
            ("E16", 0),   # Hard neg: DevOps -- Terraform/Jenkins, zero ML
        ],
    },
]

# ── Metric functions ────────────────────────────────────────────────
def _dcg_at_k(rel_list, k):
    return sum(r / math.log2(i + 2) for i, r in enumerate(rel_list[:k]))

def _ndcg_at_k(rel_list, k):
    ideal = _dcg_at_k(sorted(rel_list, reverse=True), k)
    return _dcg_at_k(rel_list, k) / ideal if ideal else 0.0

def _mrr(rel_list):
    for i, r in enumerate(rel_list):
        if r > 0: return 1.0 / (i + 1)
    return 0.0

def _map_at_k(rel_list, k):
    """Mean Average Precision -- binary hit (rel > 0), averaged over positions."""
    hits = prec_sum = 0
    n_rel = sum(1 for r in rel_list if r > 0)
    for i, r in enumerate(rel_list[:k]):
        if r > 0:
            hits += 1
            prec_sum += hits / (i + 1)
    return prec_sum / min(k, n_rel) if hits else 0.0


def run_per_query_ir_eval(score_fn, system_name, K=3) -> dict:
    """
    Per-query IR evaluation over IR_QUERIES.

    For each query:
      1. Score every candidate resume against the query job description.
      2. Sort candidates descending by score.
      3. Compute NDCG@K, MRR, MAP@K from the graded relevance labels.

    Returns macro-averaged metrics + per-query breakdown for diagnosis.
    """
    query_results = []
    resume_lookup = {e["id"]: e["resume"] for e in EVAL_DATASET}

    for q in IR_QUERIES:
        job = q["job"]
        scored = []
        for eid, rel in q["candidates"]:
            s = score_fn(resume_lookup[eid], job)
            scored.append((s, rel, eid))
        scored.sort(key=lambda x: x[0], reverse=True)
        rel_list = [r for _, r, _ in scored]

        query_results.append({
            "qid":     q["qid"],
            "label":   q["label"],
            "ndcg_k":  _ndcg_at_k(rel_list, K),
            "mrr":     _mrr(rel_list),
            "ap_k":    _map_at_k(rel_list, K),
            "ranking": [(eid, round(s, 4), r) for s, r, eid in scored],
        })

    n = len(query_results)
    return {
        "System":     system_name,
        f"NDCG@{K}": round(sum(r["ndcg_k"] for r in query_results) / n, 3),
        "MRR":        round(sum(r["mrr"]    for r in query_results) / n, 3),
        f"MAP@{K}":  round(sum(r["ap_k"]   for r in query_results) / n, 3),
        "_per_query": query_results,
    }


# ── Run evaluation ────────────────────────────────────────────────
console.rule("[bold blue]IR Retrieval Evaluation — Per-Query (Redesigned)[/bold blue]")
K = 3

ir_results = [
    run_per_query_ir_eval(
        lambda r, j: float(np.dot(INGESTION.embed(r), INGESTION.embed(j))),
        "Cosine (baseline)", K,
    ),
    run_per_query_ir_eval(
        lambda r, j: TRAINER.predict(INGESTION.embed(r), INGESTION.embed(j)),
        "Calibrated Ranker", K,
    ),
    run_per_query_ir_eval(
        lambda r, j: (
            0.30 * float(np.dot(INGESTION.embed(r), INGESTION.embed(j))) +
            0.55 * TRAINER.predict(INGESTION.embed(r), INGESTION.embed(j)) +
            0.15 * (1.0 if j[:100] in {d.description[:100] for d, _ in CORPUS.retrieve(INGESTION.embed(r))} else 0.0)
        ),
        "Ensemble (Full)", K,
    ),
]

# ── Macro summary table ──────────────────────────────────────────────
summary = Table(
    title=f"IR Evaluation — Macro-Averaged over {len(IR_QUERIES)} Queries (K={K})",
    header_style="bold cyan",
)
summary.add_column("System",    min_width=20)
summary.add_column(f"NDCG@{K}", justify="right")
summary.add_column("MRR",       justify="right")
summary.add_column(f"MAP@{K}",  justify="right")
for r in ir_results:
    summary.add_row(r["System"], str(r[f"NDCG@{K}"]), str(r["MRR"]), str(r[f"MAP@{K}"]))
console.print(summary)

# ── Per-query breakdown ────────────────────────────────────────────────
console.print()
breakdown = Table(
    title="Per-Query NDCG@3 Breakdown  (Δ = Ensemble − Cosine)",
    header_style="bold",
    show_lines=True,
)
breakdown.add_column("Query",        min_width=20)
breakdown.add_column("Pool",         justify="center", min_width=4)
for r in ir_results:
    breakdown.add_column(r["System"], justify="right", min_width=10)
breakdown.add_column("Δ (Ens−Cos)", justify="right", min_width=10)

cosine_pq   = {q["qid"]: q["ndcg_k"] for q in ir_results[0]["_per_query"]}
ranker_pq   = {q["qid"]: q["ndcg_k"] for q in ir_results[1]["_per_query"]}
ensemble_pq = {q["qid"]: q["ndcg_k"] for q in ir_results[2]["_per_query"]}

for q in IR_QUERIES:
    qid   = q["qid"]
    pool  = len(q["candidates"])
    cos   = cosine_pq[qid]
    rnk   = ranker_pq[qid]
    ens   = ensemble_pq[qid]
    delta = ens - cos
    delta_str = (
        f"[green]+{delta:.3f}[/green]" if delta >  0.005 else
        f"[red]{delta:.3f}[/red]"      if delta < -0.005 else
        f"[dim]{delta:+.3f}[/dim]"
    )
    breakdown.add_row(
        f"{qid} {q['label']}",
        str(pool),
        f"{cos:.3f}", f"{rnk:.3f}", f"{ens:.3f}",
        delta_str,
    )
console.print(breakdown)
console.print(
    f"\n[dim]Candidate pools contain only same-domain confusables — no cross-domain easy negatives. "
    f"Relevance is graded: 2=strong match, 1=near-miss, 0=hard negative. "
    f"K={K}, {len(IR_QUERIES)} queries, macro-averaged.[/dim]"
)


──────────────────────────────── IR Retrieval Evaluation — Per-Query (Redesigned) ─────────────────────────────────

 IR Evaluation — Macro-Averaged over 8 Queries 
                     (K=3)                     
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━┳━━━━━━━┓
┃ System               ┃ NDCG@3 ┃ MRR ┃ MAP@3 ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━╇━━━━━━━┩
│ Cosine (baseline)    │  0.966 │ 1.0 │ 0.979 │
│ Calibrated Ranker    │  0.953 │ 1.0 │ 0.938 │
│ Ensemble (Full)      │   0.97 │ 1.0 │ 0.979 │
└──────────────────────┴────────┴─────┴───────┘

                           Per-Query NDCG@3 Breakdown  (Δ = Ensemble − Cosine)                           
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Query                  ┃ Pool ┃ Cosine (baseline) ┃ Calibrated Ranker ┃ Ensemble (Full) ┃ Δ (Ens−Cos) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ Q01 ML Recommendations │  6   │             1.000 │             1.000 │           1.000 │      +0.000 │
├────────────────────────┼──────┼───────────────────┼───────────────────┼─────────────────┼─────────────┤
│ Q02 AI Research Eng    │  5   │             0.965 │             0.867 │           1.000 │      +0.035 │
├────────────────────────┼──────┼───────────────────┼───────────────────┼─────────────────┼─────────────┤
│ Q03 MLOps Engineer     │  4   │             1.000 │             1.000 │           1.000 │      +0.000 │
├────────────────────────┼──────┼───────────────────┼───────────────────┼─────────────────┼─────────────┤
│ Q04 ML Platform Eng    │  4   │             1.000 │             1.000 │           1.000 │      +0.000 │
├────────────────────────┼──────┼───────────────────┼───────────────────┼─────────────────┼─────────────┤
│ Q05 AI Infra Engineer  │  4   │             1.000 │             1.000 │           1.000 │      +0.000 │
├────────────────────────┼──────┼───────────────────┼───────────────────┼─────────────────┼─────────────┤
│ Q06 Computer Vision    │  4   │             1.000 │             1.000 │           1.000 │      +0.000 │
├────────────────────────┼──────┼───────────────────┼───────────────────┼─────────────────┼─────────────┤
│ Q07 Data Engineer      │  4   │             1.000 │             1.000 │           1.000 │      +0.000 │
├────────────────────────┼──────┼───────────────────┼───────────────────┼─────────────────┼─────────────┤
│ Q08 Junior ML Engineer │  4   │             0.760 │             0.760 │           0.760 │      +0.000 │
└────────────────────────┴──────┴───────────────────┴───────────────────┴─────────────────┴─────────────┘

Candidate pools contain only same-domain confusables — no cross-domain easy negatives. Relevance is graded: 
2=strong match, 1=near-miss, 0=hard negative. K=3, 8 queries, macro-averaged.

---
## 📝 CELL 17 — Final Output

In [21]:
console.rule("[bold green]Final Tailored Resume[/bold green]")

# Re-rank accepted bullets by ranker score (JD relevance)
jd_emb_rerank = INGESTION.embed(JOB_DESCRIPTION)
all_bullets = [eb for eb in OPT_RESUME.evidenced_bullets if not eb.rejected]
bullet_scores = []
for eb in all_bullets:
    b_emb = INGESTION.embed(eb.bullet)
    jd_rel_score = float(np.dot(b_emb, jd_emb_rerank))
    bullet_scores.append((eb, jd_rel_score))
# Sort descending by JD relevance; top bullets first
passed_bullets = [eb for eb, _ in sorted(bullet_scores, key=lambda x: x[1], reverse=True)]
console.print(f"[dim]Re-ranked {len(passed_bullets)} bullets by JD relevance score[/dim]")
console.print(Panel(
    f"[bold]{OPT_RESUME.candidate_name}[/bold]\n\n"
    f"[italic]{OPT_RESUME.summary}[/italic]\n\n"
    + "\n".join(f"• {eb.bullet}  [dim][p={eb.p_hallucination:.2f}][/dim]" for eb in passed_bullets),
    title=f"🎯 Tailored Resume — {OPT_RESUME.target_role[:50]}",
    border_style="green"
))

console.print(Panel(OPT_RESUME.cover_letter, title="📨 Cover Letter", border_style="blue"))

# Latency
if LATENCY_LOG:
    lat_df = pd.DataFrame(LATENCY_LOG).groupby("component")["ms"].agg(["mean","count"]).round(1)
    lat_df.columns = ["Avg ms", "Calls"]
    console.print("\n[bold]Latency Profile:[/bold]")
    console.print(lat_df.to_string())


────────────────────────────────────────────── Final Tailored Resume ──────────────────────────────────────────────

Re-ranked 8 bullets by JD relevance score

╭──────────────────── 🎯 Tailored Resume —  AI Engineer I | American Express — Agentic AI  Yo ────────────────────╮
│ Bassam Nazer                                                                                                    │
│                                                                                                                 │
│ AI/ML Engineer with experience deploying production deep learning systems, including a multi-label chest X-ray  │
│ classification pipeline field-deployed in mobile TB screening ambulances. Built multiple agentic AI and RAG     │
│ systems, including a LangGraph-based agent with tool-calling and a hierarchical retrieval pipeline over 10,000+ │
│ indexed chunks that reduced irrelevant context by ~40%.                                                         │
│                                                                                                                 │
│ • Built a LangGraph-based agentic pipeline orchestrating three sequential tools (PubMed API search, FAISS       │
│ indexing, semantic retrieval) with stateful routing and an enforced execution order.                            │
│ • Designed a FAISS retrieval layer with nomic-embed-text embeddings over PubMed abstracts, constraining         │
│ generation to retrieved context only with a fault-tolerant tool dispatcher and JSON fallback.                   │
│ • Designed and trained a DenseNet201 deep learning pipeline for multi-label chest X-ray classification across   │
│ 14 pulmonary pathologies, achieving 0.92 recall and 0.86 specificity on a held-out test set.                    │
│ • Reduced hallucination rate from ~28% to 0% by building a two-stage optimization pipeline that generates       │
│ evidence-linked bullets constrained strictly to facts from a typed JSON FactSheet.                              │
│ • Developed a two-stage hierarchical retrieval pipeline over 100+ PDFs and 10,000+ indexed chunks, reducing     │
│ irrelevant context by ~40% versus a flat single-stage retrieval baseline.                                       │
│ • Addressed severe class imbalance using focal loss and weighted binary cross-entropy by deploying an inference │
│ pipeline with result caching via Flask REST APIs on NVIDIA Quadro multi-GPU systems; system is now undergoing   │
│ clinical validation by ICMR–NIRT radiologists and field-deployed in National Health Mission Chennai’s mobile TB │
│ screening ambulances for rural population screening.                                                            │
│ • Trained a learned binary hallucination detector using a 4-feature sklearn MLP on self-supervised synthetic    │
│ pairs, achieving an F1 score of 0.80 on a curated 15-example validation set.                                    │
│ • Engineered a PyTorch pairwise contrastive ranker over a 40-pair IR benchmark, where the ensemble scorer       │
│ achieved NDCG@5 = 1.0 and Precision@5 = 1.0.                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📨 Cover Letter ────────────────────────────────────────────────╮
│ I am writing to express my strong interest in the AI Engineer I position focused on Agentic AI at American      │
│ Express. My experience in developing and deploying production AI/ML systems, combined with hands-on projects    │
│ building agentic workflows and RAG pipelines, aligns directly with the responsibilities for building production │
│ agentic AI systems.                                                                                             │
│                                                                                                                 │
│ I have built a LangGraph-based agentic pipeline that orchestrates three sequential tools, including a PubMed    │
│ API search and FAISS indexing, using stateful routing to manage execution order. In another project, I          │
│ developed a two-stage hierarchical RAG system over 10,000+ indexed chunks that reduced irrelevant context by    │
│ approximately 40%. My work also includes implementing constrained generation, hallucination detection that      │
│ achieved an F1 score of 0.80, and deploying production inference pipelines via Flask REST APIs.                 │
│                                                                                                                 │
│ My professional experience as a Project Associate at the Centre for Development of Advanced Computing (C-DAC)   │
│ involved designing and deploying a production medical imaging system that is currently field-deployed for rural │
│ population screening. I am confident that my skills in building reliable, grounded, and production-ready AI     │
│ systems would enable me to contribute effectively to your team. Thank you for your time and consideration.      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Latency Profile:

Avg ms  Calls
component                           
embed                    47.8    930
explainer.attribute    2039.1      1
optimizer.generate    35815.0      1
optimizer.regenerate  18307.4      3
parser.extract        55489.4      1
rag.retrieve              0.1     36
scorer.score           8062.5      1